In [3]:
## prerequisites
# Installiert alle benötigten Bibliotheken in den exakten Versionen für maximale Reproduzierbarkeit.

# 1. Standard-ML-Bibliotheken (Scikit-Learn für ML-Baselines, Torch für Deep Learning)
%pip install scikit-learn
%pip install torch

# 2. Exakte Versionen aus der requirements.txt (Datenverarbeitung & API-Kommunikation)
%pip install pymupdf
%pip install requests
%pip install numpy
%pip install langchain-text-splitters
%pip install pymupdf4llm
%pip install matplotlib

# 3. Spezialisierte Erweiterungen (Wichtig für Multi-Label Cross-Validation)
%pip install iterative-stratification

# 4. Implementierung der Ladebalken
%pip install tqdm ipywidgets
%pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated pa

In [1]:
   %run -i 01_cockpit.py

/home/Mike-bazz/.local/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Projektstruktur wird initialisiert
Quelle: pdf_raw | Embedding: mxbai-embed-large | LLM: llama3.2
Chunking: recursive_1200c_200o_norefs | Pooling: max | Seed: 42
Vorhersageziele: 9 Spalten

Daten-Audit
Label-Datei: IMWUT_Dataset_original.csv (SHA-256 ee5cf2727b11...)

ID-Nummerierung je Jahr
2023: 1..205, 205 Eintraege, lueckenlos
2024: 1..209, 208 Eintraege, bekannte Luecken [165]
2025: 1..243, 242 Eintraege, bekannte Luecken [173]
Ausgeschlossen: 2023_Dec_188 (Defektes Font-Encoding, Text nicht extrahierbar; aus Tabelle entfernt)

Zusammenfassung Datenbasis
Theoretisches Maximum (ID-Zaehler): 657
Annotiert:                          655
Verwendet:                          654
Vorhandene PDFs (verwendet):        654
Fehlende PDFs:                      0
PDFs ohne Label-Eintrag:            0

Label-Mapping
code_required: 4 Eintraege umkodiert
data_required: 6 Eintraege umkodiert
hardware_required: 60 Eintraege umkodiert
software_required: 68 Eintraege umkodiert
code_available: 84 Eintra

In [10]:
sort_and_rename_raw_pdfs()

Sortiere PDFs: 100%|██████████████████████████| 654/654 [00:28<00:00, 23.09it/s]


Zugeordnet und kopiert: 654
Nicht zuordenbar:       0


In [8]:
"""
Segment 2: PDF-zu-Markdown-Konvertierung.

Konvertiert die PDFs aller verwendeten Studien (master_label_df) mit pymupdf4llm.
Schreibt ein Manifest (Konverter-Version, MuPDF-Warnungen, Hash pro Datei) und
einen Korpus-Fingerprint. Die Qualitaetsbewertung (U+FFFD, Textmenge) erfolgt
im Notebook Datenvalidierung. Laeuft nur bei DATEN_QUELLE == "markdown".

Fortsetzung nach Abbruch: Jeder Lauf legt eine Statusdatei an und protokolliert
fertige Dateien sofort. Existiert die Statusdatei beim Start noch, wurde der
vorige Lauf abgebrochen; das Segment setzt ihn bei der ersten offenen Datei fort.
"""

from importlib.metadata import version

import pymupdf4llm

FORCE_RECONVERT = False   # True: vorhandene .md-Dateien neu erzeugen

MANIFEST_PATH = MARKDOWN_ROOT / "conversion_manifest.csv"
STATUS_PATH = MARKDOWN_ROOT / "conversion_lauf.json"            # existiert nur waehrend eines Laufs
FORTSCHRITT_PATH = MARKDOWN_ROOT / "conversion_progress.jsonl"  # eine Zeile pro fertiger Datei

CONVERTER_VERSION = f"pymupdf4llm {version('pymupdf4llm')} / pymupdf {version('pymupdf')}"

# MuPDF-Meldungen nicht auf die Konsole schreiben (stoert tqdm),
# sondern pro Datei einsammeln und ins Manifest uebernehmen.
fitz.TOOLS.mupdf_display_errors(False)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def lauf_starten_oder_fortsetzen() -> tuple[dict, dict]:
    """
    Liefert (Lauf, bereits erledigte Eintraege dieses Laufs).
    Ein abgebrochener Lauf wird nur fortgesetzt, wenn die Konverter-Version
    gleich geblieben ist; sonst entstuende ein Korpus aus gemischten Versionen.
    """
    if STATUS_PATH.exists():
        lauf = json.loads(STATUS_PATH.read_text(encoding="utf-8"))
        if lauf["converter"] == CONVERTER_VERSION:
            erledigt = {}
            if FORTSCHRITT_PATH.exists():
                for zeile in FORTSCHRITT_PATH.read_text(encoding="utf-8").splitlines():
                    try:
                        eintrag = json.loads(zeile)
                    except json.JSONDecodeError:
                        continue  # beim Abbruch unvollstaendig geschriebene Zeile
                    if eintrag.get("lauf_id") == lauf["lauf_id"]:
                        erledigt[eintrag["paper_id"]] = eintrag
            print(f"Abgebrochener Lauf {lauf['lauf_id']} erkannt: "
                  f"{len(erledigt)} Dateien erledigt, setze fort (force={lauf['force']}).")
            return lauf, erledigt
        print(f"Abgebrochener Lauf mit anderer Konverter-Version ({lauf['converter']}) verworfen.")

    lauf = {
        "lauf_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
        "converter": CONVERTER_VERSION,
        "force": FORCE_RECONVERT,
    }
    STATUS_PATH.write_text(json.dumps(lauf), encoding="utf-8")
    FORTSCHRITT_PATH.unlink(missing_ok=True)
    return lauf, {}


def protokolliere(eintrag: dict) -> None:
    """Haengt einen fertigen Eintrag sofort an die Fortschrittsdatei an."""
    with open(FORTSCHRITT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(eintrag) + "\n")


if DATEN_QUELLE == "markdown":
    print(f"Markdown-Konvertierung mit {CONVERTER_VERSION}")

    # Reste eines abgebrochenen Schreibvorgangs entfernen
    for tmp in MARKDOWN_ROOT.rglob("*.md.tmp"):
        tmp.unlink()
        print(f"Unvollstaendige Datei entfernt: {tmp.name}")

    lauf, erledigt = lauf_starten_oder_fortsetzen()

    # Version und Warnungen frueherer Laeufe fuer uebersprungene Dateien
    alt_manifest = {}
    if MANIFEST_PATH.exists():
        alt_df = pd.read_csv(MANIFEST_PATH)
        alt_manifest = alt_df.set_index("paper_id")[["converter", "n_mupdf_warnungen"]].to_dict("index")

    manifest = []
    for row in tqdm(master_label_df[["paper_id", "year"]].itertuples(index=False),
                    total=len(master_label_df), desc="PDF -> MD"):
        if row.paper_id in erledigt:
            manifest.append(erledigt[row.paper_id])
            continue

        pdf_path = STUDIEN_ROOT / row.year / f"{row.paper_id}.pdf"
        md_path = MARKDOWN_ROOT / row.year / f"{row.paper_id}.md"
        eintrag = {"lauf_id": lauf["lauf_id"], "paper_id": row.paper_id, "year": row.year}

        if not pdf_path.exists():
            eintrag["status"] = "pdf_fehlt"
        elif md_path.exists() and not lauf["force"]:
            alt = alt_manifest.get(row.paper_id, {})
            eintrag.update(status="uebersprungen",
                           converter=alt.get("converter", "unbekannt"),
                           n_mupdf_warnungen=alt.get("n_mupdf_warnungen"))
        else:
            fitz.TOOLS.reset_mupdf_warnings()
            try:
                text = pymupdf4llm.to_markdown(str(pdf_path))
            except Exception as e:
                eintrag.update(status="fehler", fehler=str(e))
            else:
                # Atomar schreiben: Abbruch hinterlaesst keine halbe .md-Datei
                tmp_path = md_path.with_suffix(".md.tmp")
                tmp_path.write_text(text, encoding="utf-8")
                tmp_path.replace(md_path)
                warnungen = fitz.TOOLS.mupdf_warnings(reset=True)
                eintrag.update(status="konvertiert",
                               converter=CONVERTER_VERSION,
                               n_mupdf_warnungen=len(warnungen.splitlines()) if warnungen else 0)

        if eintrag["status"] in ("konvertiert", "uebersprungen"):
            text = md_path.read_text(encoding="utf-8")
            eintrag.update(chars=len(text), md_sha256=sha256_text(text))

        manifest.append(eintrag)
        protokolliere(eintrag)

    manifest_df = pd.DataFrame(manifest)
    manifest_df.to_csv(MANIFEST_PATH, index=False)

    # Lauf vollstaendig: Statusdateien entfernen
    STATUS_PATH.unlink(missing_ok=True)
    FORTSCHRITT_PATH.unlink(missing_ok=True)

    print("\nStatus:")
    print(manifest_df["status"].value_counts().to_string())

    versionen = manifest_df["converter"].dropna().unique()
    if len(versionen) > 1:
        print(f"WARNUNG: Korpus mit mehreren Konverter-Versionen erzeugt: {list(versionen)}")

    for status in ["pdf_fehlt", "fehler"]:
        betroffen = manifest_df.loc[manifest_df["status"] == status, "paper_id"].tolist()
        if betroffen:
            print(f"{status}: {betroffen}")

    ok = manifest_df.dropna(subset=["md_sha256"])

    # Korpus-Fingerprint: aendert sich, sobald sich eine einzige .md-Datei aendert
    MARKDOWN_CORPUS_SHA256 = hashlib.sha256(
        "".join(ok.sort_values("paper_id")["md_sha256"]).encode()
    ).hexdigest()
    print(f"\nKorpus-Fingerprint (SHA-256): {MARKDOWN_CORPUS_SHA256}")
    print(f"Manifest: {MANIFEST_PATH}")

else:
    print(f"Markdown-Konvertierung uebersprungen (DATEN_QUELLE = '{DATEN_QUELLE}').")


Markdown-Konvertierung uebersprungen (DATEN_QUELLE = 'pdf_raw').


In [4]:
"""
Segment 3: Vektorisierung der Dokumente.

Zerlegt jede verwendete Studie (master_label_df) in Chunks, erzeugt pro Chunk
einen Embedding-Vektor ueber Ollama und speichert ihn als JSON.
Ein Manifest haelt pro Studie den Hash der Quelldatei, der Embedding-Konfiguration
und den Modell-Digest fest, dazu die Lage des entfernten Literaturverzeichnisses.
Vorhandene Embeddings werden nur uebersprungen, wenn alles uebereinstimmt; eine
geaenderte Erkennungsregel fuehrt so nur bei den betroffenen Studien zu neuen Embeddings.

Fortsetzung nach Abbruch: Jedes fertige Dokument wird sofort protokolliert.
Existiert die Statusdatei beim Start noch, wurde der vorige Lauf abgebrochen;
fertige Dokumente werden uebersprungen, das unterbrochene wird neu vektorisiert.
"""

STUDIEN_LIMIT = None    # Ganzzahl nur zum Testen, None = alle Studien

QUELLE_DIR, DATEI_ENDUNG = (
    (MARKDOWN_ROOT, ".md") if DATEN_QUELLE == "markdown" else (STUDIEN_ROOT, ".pdf")
)
EMBED_MANIFEST_PATH = CURRENT_EMBEDDING_PATH / "embedding_manifest.csv"
EMBED_STATUS_PATH = CURRENT_EMBEDDING_PATH / "embedding_lauf.json"            # existiert nur waehrend eines Laufs
EMBED_FORTSCHRITT_PATH = CURRENT_EMBEDDING_PATH / "embedding_progress.jsonl"  # eine Zeile pro fertigem Dokument

# Alles, was die Vektoren beeinflusst. Aendert sich ein Wert, aendert sich der Hash.
EMBED_CONFIG = {
    "daten_quelle": DATEN_QUELLE,
    "modell": MODELL_NAME,
    "prefix": DOCUMENT_PREFIX,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "separators": CHUNK_SEPARATORS,
    "nachteilen": {"regel": "rekursiv halbieren, Overlap proportional", "min_chars": NACHTEIL_MIN_CHARS},
    "references_entfernen": REFERENCES_ENTFERNEN,
}
EMBED_CONFIG_SHA256 = hashlib.sha256(json.dumps(EMBED_CONFIG, sort_keys=True).encode()).hexdigest()


def lade_manifest() -> dict:
    """Manifest des letzten vollstaendigen Laufs plus Fortschritt eines abgebrochenen Laufs."""
    eintraege = {}
    if EMBED_MANIFEST_PATH.exists():
        eintraege = pd.read_csv(EMBED_MANIFEST_PATH).set_index("paper_id").to_dict("index")
    if EMBED_STATUS_PATH.exists():
        lauf = json.loads(EMBED_STATUS_PATH.read_text(encoding="utf-8"))
        n_fertig = 0
        if EMBED_FORTSCHRITT_PATH.exists():
            for zeile in EMBED_FORTSCHRITT_PATH.read_text(encoding="utf-8").splitlines():
                try:
                    eintrag = json.loads(zeile)
                except json.JSONDecodeError:
                    continue  # beim Abbruch unvollstaendig geschriebene Zeile
                eintraege[eintrag.pop("paper_id")] = eintrag
                n_fertig += 1
        print(f"Abgebrochener Lauf vom {lauf['gestartet']} erkannt: "
              f"{n_fertig} Dokumente fertig, setze fort.")
    return eintraege


def protokolliere(eintrag: dict) -> None:
    """Haengt ein fertiges Dokument sofort an die Fortschrittsdatei an."""
    with open(EMBED_FORTSCHRITT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(eintrag) + "\n")


def modell_digest() -> str:
    """Digest des Embedding-Modells laut Ollama. MODELL_NAME darf mit oder ohne Tag angegeben sein."""
    kandidaten = {MODELL_NAME, f"{MODELL_NAME}:latest"}
    for name, digest in get_ollama_model_info().items():
        if name in kandidaten:
            return digest
    return "unbekannt"


if not check_ollama():
    print("FEHLER: Ollama-Server nicht erreichbar.")
else:
    digest = modell_digest()
    if digest == "unbekannt":
        raise RuntimeError(f"Modell '{MODELL_NAME}' ist in Ollama nicht installiert: ollama pull {MODELL_NAME}")
    print(f"Quelle: {QUELLE_DIR} ({DATEI_ENDUNG}) | Modell: {MODELL_NAME} ({digest[:12]})")
    print(f"Ziel:   {CURRENT_EMBEDDING_PATH}")

    # Reste eines abgebrochenen Schreibvorgangs entfernen
    for tmp in CURRENT_EMBEDDING_PATH.rglob("*.json.tmp"):
        tmp.unlink()
        print(f"Unvollstaendige Datei entfernt: {tmp.name}")

    CURRENT_EMBEDDING_PATH.mkdir(parents=True, exist_ok=True)
    alt_manifest = lade_manifest()
    if not EMBED_STATUS_PATH.exists():
        EMBED_STATUS_PATH.write_text(
            json.dumps({"gestartet": datetime.now().isoformat(timespec="seconds")}), encoding="utf-8")
        EMBED_FORTSCHRITT_PATH.unlink(missing_ok=True)

    studien = master_meta_df[["paper_id", "year", "doi_url"]]
    if STUDIEN_LIMIT is not None:
        studien = studien.head(STUDIEN_LIMIT)

    manifest, fehler = {}, []
    zaehler = {"neu": 0, "uebersprungen": 0, "fehler": 0}

    pbar = tqdm(studien.itertuples(index=False), total=len(studien), desc="Vektorisierung")
    for row in pbar:
        pbar.set_postfix_str(row.paper_id)
        quelle = QUELLE_DIR / row.year / f"{row.paper_id}{DATEI_ENDUNG}"
        out_path = CURRENT_EMBEDDING_PATH / row.year / f"{row.paper_id}_embeddings.json"

        if not quelle.exists():
            fehler.append((row.paper_id, "Quelldatei fehlt"))
            zaehler["fehler"] += 1
            continue

        quelle_sha = file_sha256(quelle)
        # Chunking ist schnell; die Schnittstelle des Literaturverzeichnisses wird bei jedem
        # Lauf neu bestimmt, damit eine geaenderte Erkennungsregel erkannt wird
        chunks, metadata_internal = extract_pdf_structure(quelle)
        schnitt = schnitt_signatur(metadata_internal)

        alt = alt_manifest.get(row.paper_id, {})
        aktuell = (out_path.exists()
                   and alt.get("source_sha256") == quelle_sha
                   and alt.get("config_sha256") == EMBED_CONFIG_SHA256
                   and alt.get("model_digest") == digest)
        if aktuell:
            alt_schnitt = alt.get("references_schnitt")
            if not isinstance(alt_schnitt, str):
                # Eintraege aus Laeufen vor dieser Pruefung: Schnitt aus der Embedding-Datei lesen
                alt_meta = json.loads(out_path.read_text(encoding="utf-8"))["metadata"].get("header_info", {})
                alt_schnitt = schnitt_signatur(alt_meta)
            if alt_schnitt == schnitt:
                manifest[row.paper_id] = {**alt, "references_schnitt": schnitt}
                zaehler["uebersprungen"] += 1
                continue

        if not chunks:
            fehler.append((row.paper_id, "keine Chunks"))
            zaehler["fehler"] += 1
            continue

        # Zu lange Chunks werden in embed_chunk nachgeteilt; die Teile erhalten
        # fortlaufende chunk_ids und verweisen auf ihren Ursprungs-Chunk.
        # Alle Chunks oder keiner: ein unvollstaendiges Dokument wuerde
        # beim naechsten Lauf als fertig gelten und nie nachgeholt.
        final_chunks, n_fehlend, n_nachgeteilt = [], 0, 0
        for item in chunks:
            teile = embed_chunk(item["text"])
            if teile is None:
                n_fehlend += 1
                continue
            n_nachgeteilt += teile[0]["nachgeteilt"]
            for teil in teile:
                final_chunks.append({
                    "chunk_id": len(final_chunks),
                    "parent_chunk_id": item["chunk_id"],
                    **teil,
                })
        if n_fehlend:
            fehler.append((row.paper_id, f"{n_fehlend} von {len(chunks)} Chunks ohne Vektor"))
            zaehler["fehler"] += 1
            continue
        chunks = final_chunks

        zeitpunkt = datetime.now().isoformat(timespec="seconds")
        daten = {
            "metadata": {
                "paper_id": row.paper_id,
                "source_filename": f"{row.paper_id}.pdf",
                "doi": str(row.doi_url).replace("https://doi.org/", "").lower(),
                "header_info": metadata_internal,
                "model": MODELL_NAME,
                "model_digest": digest,
                "strategy": STRATEGIE_NAME,
                "daten_quelle": DATEN_QUELLE,
                "embed_config": EMBED_CONFIG,
                "source_sha256": quelle_sha,
                "timestamp": zeitpunkt,
            },
            "chunks": chunks,
        }
        # Atomar schreiben: Abbruch hinterlaesst keine halbe JSON-Datei
        out_path.parent.mkdir(parents=True, exist_ok=True)
        tmp_path = out_path.with_suffix(".json.tmp")
        tmp_path.write_text(json.dumps(daten), encoding="utf-8")
        tmp_path.replace(out_path)

        eintrag = {
            "year": row.year,
            "n_chunks": len(chunks),
            "n_nachgeteilt": n_nachgeteilt,
            "references_schnitt": schnitt,
            "source_sha256": quelle_sha,
            "config_sha256": EMBED_CONFIG_SHA256,
            "model_digest": digest,
            "timestamp": zeitpunkt,
        }
        manifest[row.paper_id] = eintrag
        protokolliere({"paper_id": row.paper_id, **eintrag})
        zaehler["neu"] += 1
    pbar.close()

    # Eintraege nicht bearbeiteter Studien (z. B. bei STUDIEN_LIMIT) behalten
    gesamt = {**alt_manifest, **manifest}
    manifest_df = (pd.DataFrame.from_dict(gesamt, orient="index")
                   .rename_axis("paper_id").reset_index())
    manifest_df.to_csv(EMBED_MANIFEST_PATH, index=False)

    # Lauf vollstaendig: Statusdateien entfernen
    EMBED_STATUS_PATH.unlink(missing_ok=True)
    EMBED_FORTSCHRITT_PATH.unlink(missing_ok=True)

    print(f"\nNeu: {zaehler['neu']} | Uebersprungen: {zaehler['uebersprungen']} | Fehler: {zaehler['fehler']}")
    for p_id, grund in fehler:
        print(f"  {p_id}: {grund}")

    # Kennzahlen nur ueber die Studien dieses Laufs
    if manifest:
        lauf_df = pd.DataFrame.from_dict(manifest, orient="index").rename_axis("paper_id").reset_index()
        EMBEDDING_CORPUS_SHA256 = hashlib.sha256(
            "".join(lauf_df.sort_values("paper_id")["source_sha256"]).encode()
            + EMBED_CONFIG_SHA256.encode()
        ).hexdigest()
        print(f"Chunks gesamt: {int(lauf_df['n_chunks'].sum())}")
        if "n_nachgeteilt" in lauf_df:
            n_dok = int((lauf_df["n_nachgeteilt"] > 0).sum())
            print(f"Nachgeteilte Chunks: {int(lauf_df['n_nachgeteilt'].sum())} in {n_dok} Dokumenten")
        print(f"Embedding-Fingerprint (Quellen + Konfiguration): {EMBEDDING_CORPUS_SHA256}")

Quelle: /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/pdfs (.pdf) | Modell: mxbai-embed-large (468836162de7)
Ziel:   /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/embeddings/pdf_raw/mxbai-embed-large/recursive_1200c_200o_norefs


Vektorisierung: 100%|███████████| 654/654 [02:02<00:00,  5.36it/s, 2025_Dec_243]


Neu: 0 | Uebersprungen: 654 | Fehler: 0
Chunks gesamt: 63455
Nachgeteilte Chunks: 530 in 176 Dokumenten
Embedding-Fingerprint (Quellen + Konfiguration): 61685410c21f243df44a6afe1e346de413112ee1d9af2fd0b26b928d333003a8


In [5]:
"""
Segment 4: Fusion von Embeddings und Labels zu Trainingsmatrizen.
 
Fasst die Chunk-Vektoren jeder Studie per Pooling zu einem Dokumentvektor
zusammen (X) und kodiert die Ziele aus dem Klassifikationsdatensatz binaer (y).
Alle Pooling-Strategien werden in einem Durchlauf erzeugt; die im Cockpit
gewaehlte (POOLING_STRATEGIE) steht danach als X, y im Speicher.
 
Reihenfolge der Zeilen = sortierte paper_ids aus master_label_df. Sie ist damit
unabhaengig vom Dateisystem und wird als paper_ids.json mitgespeichert, damit
jede Vorhersage einer Studie zugeordnet werden kann.
"""
 
ERLAUBE_UNVOLLSTAENDIG = False   # True: fehlende Embeddings ueberspringen statt abzubrechen
 
ZUSATZ_TARGETS = [t for t in PREDICTION_TARGETS if t != "paper_type"]
ML_COLUMNS = PAPER_TYPE_CATEGORIES + ZUSATZ_TARGETS   # 6 Paper-Typen + 8 binaere Ziele
 
POOLING_FUNKTIONEN = {
    "mean": lambda v: v.mean(axis=0),
    "max": lambda v: v.max(axis=0),
    "mean_max_concat": lambda v: np.concatenate([v.mean(axis=0), v.max(axis=0)]),
}
 
 
def kodiere_labels(row) -> list:
    """Multi-Hot fuer paper_type, 0/1 fuer die binaeren Ziele. Unerwartete Werte sind ein Fehler."""
    typen = {t.strip() for t in str(row["paper_type"]).split(";") if t.strip()}
    unbekannt = typen - set(PAPER_TYPE_CATEGORIES)
    if unbekannt:
        raise ValueError(f"{row['paper_id']}: unbekannter paper_type {unbekannt}")
    labels = [int(kat in typen) for kat in PAPER_TYPE_CATEGORIES]
 
    for ziel in ZUSATZ_TARGETS:
        wert = row[ziel]
        if wert not in ("Yes", "No"):
            # Das Cockpit liefert nur Yes/No; alles andere deutet auf ein uebersprungenes Mapping
            raise ValueError(f"{row['paper_id']}: {ziel} = {wert!r} (erwartet Yes/No)")
        labels.append(int(wert == "Yes"))
    return labels
 
 
# Einheitlichkeit des Embedding-Korpus pruefen (eine Konfiguration, eine Modellversion)
embed_manifest_path = CURRENT_EMBEDDING_PATH / "embedding_manifest.csv"
if not embed_manifest_path.exists():
    raise FileNotFoundError(f"Embedding-Manifest fehlt: {embed_manifest_path}. Segment 3 zuerst ausfuehren.")
embed_manifest = pd.read_csv(embed_manifest_path).set_index("paper_id")
embed_manifest = embed_manifest.loc[embed_manifest.index.intersection(master_label_df["paper_id"])]
for spalte in ["config_sha256", "model_digest"]:
    werte = embed_manifest[spalte].dropna().unique()
    if len(werte) > 1:
        raise RuntimeError(f"Embeddings mit unterschiedlichen Werten fuer {spalte}: {list(werte)}")
 
pooled = {strategie: [] for strategie in POOLING_FUNKTIONEN}
y_list, paper_ids, n_chunks, fehlend = [], [], [], []
 
studien = master_label_df.sort_values("paper_id")
for row in tqdm(studien.to_dict("records"), desc="Fusion"):
    json_path = CURRENT_EMBEDDING_PATH / row["year"] / f"{row['paper_id']}_embeddings.json"
    if not json_path.exists():
        fehlend.append(row["paper_id"])
        continue
 
    daten = json.loads(json_path.read_text(encoding="utf-8"))
    vektoren = np.asarray([c["vector"] for c in daten["chunks"]], dtype=np.float32)
    if vektoren.ndim != 2 or len(vektoren) == 0:
        fehlend.append(row["paper_id"])
        continue
 
    for strategie, funktion in POOLING_FUNKTIONEN.items():
        pooled[strategie].append(funktion(vektoren))
    y_list.append(kodiere_labels(row))
    paper_ids.append(row["paper_id"])
    n_chunks.append(len(vektoren))
 
if fehlend:
    print(f"Ohne Embeddings: {len(fehlend)} Studien: {', '.join(fehlend[:20])}")
    if not ERLAUBE_UNVOLLSTAENDIG:
        raise RuntimeError("Unvollstaendige Embeddings. Segment 3 abschliessen "
                           "oder ERLAUBE_UNVOLLSTAENDIG = True setzen und in der Arbeit berichten.")
 
y_alle = np.asarray(y_list, dtype=np.int8)
 
for strategie, vektoren in pooled.items():
    ziel_dir = FUSION_BASE_ROOT / DATEN_QUELLE / strategie / MODELL_NAME / STRATEGIE_NAME
    ziel_dir.mkdir(parents=True, exist_ok=True)
    X_strategie = np.stack(vektoren)
    np.save(ziel_dir / "X_data.npy", X_strategie)
    np.save(ziel_dir / "y_data.npy", y_alle)
    (ziel_dir / "y_columns.json").write_text(json.dumps(ML_COLUMNS), encoding="utf-8")
    (ziel_dir / "paper_ids.json").write_text(json.dumps(paper_ids), encoding="utf-8")
    (ziel_dir / "fusion_manifest.json").write_text(json.dumps({
        "pooling": strategie,
        "n_studien": len(paper_ids),
        "dimensionen": int(X_strategie.shape[1]),
        "fehlende_embeddings": fehlend,
        "chunks_pro_studie": {"min": int(min(n_chunks)), "median": float(np.median(n_chunks)),
                              "max": int(max(n_chunks))},
        "embedding_config_sha256": str(embed_manifest["config_sha256"].iloc[0]),
        "model_digest": str(embed_manifest["model_digest"].iloc[0]),
        "klassifikation_csv_sha256": KLASSIFIKATION_CSV_SHA256,
        "X_sha256": hashlib.sha256(X_strategie.tobytes()).hexdigest(),
        "y_sha256": hashlib.sha256(y_alle.tobytes()).hexdigest(),
    }, indent=2), encoding="utf-8")
    print(f"{strategie:>16}: X {X_strategie.shape} -> {ziel_dir}")
 
# Im Cockpit gewaehlte Strategie fuer die folgenden Segmente bereitstellen
X = np.stack(pooled[POOLING_STRATEGIE])
y = y_alle
FUSION_PAPER_IDS = paper_ids
 
print(f"\ny: {y.shape} ({len(PAPER_TYPE_CATEGORIES)} Paper-Typen + {len(ZUSATZ_TARGETS)} binaere Ziele)")
print("Positive Klasse je Ziel:")
for spalte, anzahl in zip(ML_COLUMNS, y.sum(axis=0)):
    print(f"  {spalte:<22} {int(anzahl):>4} / {len(y)}")
 

Fusion: 100%|█████████████████████████████████| 654/654 [00:24<00:00, 26.29it/s]

            mean: X (654, 1024) -> /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/fusionsdaten/pdf_raw/mean/mxbai-embed-large/recursive_1200c_200o_norefs
             max: X (654, 1024) -> /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/fusionsdaten/pdf_raw/max/mxbai-embed-large/recursive_1200c_200o_norefs
 mean_max_concat: X (654, 2048) -> /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/fusionsdaten/pdf_raw/mean_max_concat/mxbai-embed-large/recursive_1200c_200o_norefs

y: (654, 14) (6 Paper-Typen + 8 binaere Ziele)
Positive Klasse je Ziel:
  Algorithm Development   498 / 654
  Hardware Development    174 / 654
  Software Development    351 / 654
  User Study              389 / 654
  Dataset Paper            57 / 654
  Literature Review        12 / 654
  code_required           565 / 654
  code_available          154 / 654
  code_url                171 / 654
  data_required           469 / 654
  data_available          188 / 654
  data_url                134 / 654
  hardware_req

In [6]:
"""
Segment 5: Multi-Label-Klassifikation mit scikit-learn (MLPClassifier).

Ein MLP mit 14 Sigmoid-Ausgaben (6 Paper-Typen + 8 binaere Ziele). Die Ausgaben
teilen sich die verborgenen Schichten, werden aber unabhaengig voneinander
vorhergesagt (keine explizite Abhaengigkeit zwischen den Labels).

Evaluierung:
- "cross_validation": wiederholte, multilabel-stratifizierte k-Fold-Kreuzvalidierung.
  Jede Wiederholung nutzt einen eigenen Seed fuer Split und Initialisierung;
  berichtet werden Mittelwert und Standardabweichung ueber die Wiederholungen.
- "holdout": ein multilabel-stratifizierter 80/20-Split je Wiederholung.

Early Stopping auf dem Validierungs-Loss (binaere Kreuzentropie) einer inneren,
stratifizierten Validierungsmenge; die Gewichte der besten Epoche werden
wiederhergestellt. Der eingebaute Mechanismus von scikit-learn wird bewusst
nicht verwendet: Er stoppt bei Multi-Label-Daten auf der Subset Accuracy und
teilt die Validierungsmenge nicht stratifiziert ab.
"""

import copy
from importlib.metadata import version as paket_version

import matplotlib.pyplot as plt
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
from sklearn.metrics import accuracy_score, hamming_loss, jaccard_score, precision_recall_fscore_support
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# =============================================================================
# KONFIGURATION
# =============================================================================
VALIDIERUNGS_METHODE = "cross_validation"   # "cross_validation" | "holdout"
N_FOLDS = 5
N_WIEDERHOLUNGEN = 5                        # Seeds: GLOBAL_SEED, GLOBAL_SEED + 1, ...
HOLDOUT_TESTANTEIL = 0.2

# Einzige Quelle der Hyperparameter; Report und Run-Log werden daraus erzeugt.
# Vorab festgelegt, nicht auf den Testdaten abgestimmt.
MLP_PARAMS = {
    "hidden_layer_sizes": (512, 256),
    "activation": "relu",
    "solver": "adam",
    "learning_rate_init": 5e-4,
    "alpha": 1e-4,          # L2-Regularisierung
    "batch_size": 32,
}
MAX_EPOCHEN = 600
GEDULD = 10                 # Epochen ohne Verbesserung des Validierungs-Loss
MIN_VERBESSERUNG = 1e-4
VAL_ANTEIL = 0.1            # innere Validierungsmenge fuer Early Stopping
SCHWELLE = 0.5              # Wahrscheinlichkeit ab der ein Label als positiv gilt

classifier_name = "Scikit_MLP_CrossVal" if VALIDIERUNGS_METHODE == "cross_validation" else "Scikit_MLP_Holdout"
# Feste, aus GLOBAL_SEED abgeleitete Seeds: jede Ausfuehrung nutzt dieselben Splits und Initialisierungen
SEEDS = [GLOBAL_SEED + i for i in range(N_WIEDERHOLUNGEN)]

# =============================================================================
# DATEN
# =============================================================================
X = np.load(CURRENT_FUSION_PATH / "X_data.npy")
y = np.load(CURRENT_FUSION_PATH / "y_data.npy")
ML_COLUMNS = json.loads((CURRENT_FUSION_PATH / "y_columns.json").read_text(encoding="utf-8"))
paper_ids = json.loads((CURRENT_FUSION_PATH / "paper_ids.json").read_text(encoding="utf-8"))
fusion_manifest = json.loads((CURRENT_FUSION_PATH / "fusion_manifest.json").read_text(encoding="utf-8"))

if not (len(X) == len(y) == len(paper_ids)) or y.shape[1] != len(ML_COLUMNS):
    raise ValueError("X, y, paper_ids und y_columns passen nicht zusammen. Segment 4 erneut ausfuehren.")

# Reine User Studies fuer die Sensitivitaetsanalyse (Labels folgen dort einer Protokollregel)
typ_roh = master_label_raw_df.set_index("paper_id")["paper_type"].astype(str).str.strip()
REINE_USER_STUDY = np.array([typ_roh.get(p) == "User Study" for p in paper_ids])

# =============================================================================
# RUN-ORDNER
# =============================================================================
base_dir = PREDICTIONS_ML_ROOT / classifier_name / DATEN_QUELLE / POOLING_STRATEGIE / MODELL_NAME / STRATEGIE_NAME
base_dir.mkdir(parents=True, exist_ok=True)
run_nummern = [int(d.name.split("_")[1]) for d in base_dir.glob("Run_*") if d.name.split("_")[1].isdigit()]
RUN_DIR = base_dir / f"Run_{max(run_nummern, default=0) + 1}"
RUN_DIR.mkdir()

print(f"Daten: X {X.shape}, y {y.shape} | Evaluierung: {VALIDIERUNGS_METHODE}, Seeds {SEEDS}")
print(f"Run-Ordner: {RUN_DIR}")


# =============================================================================
# TRAINING
# =============================================================================
def bce(y_wahr: np.ndarray, proba: np.ndarray) -> float:
    """Binaere Kreuzentropie, Summe ueber Labels, Mittel ueber Studien (wie MLPClassifier.loss_ ohne L2)."""
    p = np.clip(proba, 1e-7, 1 - 1e-7)
    return float(-np.mean(np.sum(y_wahr * np.log(p) + (1 - y_wahr) * np.log(1 - p), axis=1)))


def trainiere_mlp(X_tr: np.ndarray, y_tr: np.ndarray, seed: int):
    """
    Trainiert ein MLP mit Early Stopping auf dem Validierungs-Loss.
    Rueckgabe: (Modell mit besten Gewichten, Scaler, Verlauf).
    """
    innen_tr, innen_val = next(MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=VAL_ANTEIL, random_state=seed).split(X_tr, y_tr))

    # Skalierung nur auf den Trainingsdaten anpassen (kein Leakage in Validierung oder Test)
    scaler = StandardScaler().fit(X_tr[innen_tr])
    Xa, Xv = scaler.transform(X_tr[innen_tr]), scaler.transform(X_tr[innen_val])
    ya, yv = y_tr[innen_tr], y_tr[innen_val]

    mlp = MLPClassifier(**MLP_PARAMS, random_state=seed)
    verlauf = {"train_loss": [], "val_loss": []}
    bester_loss, beste_epoche, bestes_modell, ohne_verbesserung = np.inf, 0, None, 0

    for epoche in range(MAX_EPOCHEN):
        if epoche == 0:
            mlp.partial_fit(Xa, ya, classes=np.arange(ya.shape[1]))
        else:
            mlp.partial_fit(Xa, ya)
        val_loss = bce(yv, mlp.predict_proba(Xv))
        verlauf["train_loss"].append(float(mlp.loss_))   # inkl. L2-Term
        verlauf["val_loss"].append(val_loss)

        if val_loss < bester_loss - MIN_VERBESSERUNG:
            bester_loss, beste_epoche, ohne_verbesserung = val_loss, epoche, 0
            bestes_modell = copy.deepcopy(mlp)
        else:
            ohne_verbesserung += 1
            if ohne_verbesserung >= GEDULD:
                break

    verlauf["beste_epoche"] = beste_epoche
    verlauf["epochen"] = epoche + 1
    return bestes_modell, scaler, verlauf


def splits(seed: int):
    if VALIDIERUNGS_METHODE == "cross_validation":
        return MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(X, y)
    return MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=HOLDOUT_TESTANTEIL, random_state=seed).split(X, y)


def plotte_verlauf(verlauf: dict, titel: str, pfad: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(verlauf["train_loss"], label="Trainings-Loss (inkl. L2)")
    ax.plot(verlauf["val_loss"], label="Validierungs-Loss")
    ax.axvline(verlauf["beste_epoche"], color="red", linestyle="--",
               label=f"Beste Epoche ({verlauf['beste_epoche']})")
    ax.set(title=titel, xlabel="Epoche", ylabel="Binaere Kreuzentropie")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)
    fig.tight_layout()
    fig.savefig(pfad, dpi=150)
    plt.close(fig)


# =============================================================================
# EVALUIERUNG
# =============================================================================
def metriken(y_wahr: np.ndarray, y_pred: np.ndarray) -> dict:
    p, r, f, s = precision_recall_fscore_support(y_wahr, y_pred, average=None, zero_division=0)
    return {
        "precision": p, "recall": r, "f1": f, "support": s,
        "macro_f1": float(f.mean()),
        "hamming_loss": hamming_loss(y_wahr, y_pred),
        "subset_accuracy": accuracy_score(y_wahr, y_pred),
        "jaccard_samples": jaccard_score(y_wahr, y_pred, average="samples", zero_division=0),
    }


ergebnisse, oof_zeilen, verlaeufe = [], [], []

for wdh, seed in enumerate(SEEDS, start=1):
    proba = np.full(y.shape, np.nan)
    baseline = np.full(y.shape, -1, dtype=int)

    for fold, (train_idx, test_idx) in enumerate(splits(seed), start=1):
        modell, scaler, verlauf = trainiere_mlp(X[train_idx], y[train_idx], seed)
        proba[test_idx] = modell.predict_proba(scaler.transform(X[test_idx]))
        # Mehrheitsklasse je Ziel aus den Trainingsdaten des Folds
        baseline[test_idx] = (y[train_idx].mean(axis=0) >= 0.5).astype(int)

        verlaeufe.append({"wiederholung": wdh, "seed": seed, "fold": fold,
                          "beste_epoche": verlauf["beste_epoche"], "epochen": verlauf["epochen"],
                          "bester_val_loss": min(verlauf["val_loss"])})
        if wdh == 1:
            plotte_verlauf(verlauf, f"Wiederholung 1, Fold {fold}", RUN_DIR / f"lernkurve_fold-{fold}.png")
        print(f"  Wdh {wdh} Fold {fold}: beste Epoche {verlauf['beste_epoche']} von {verlauf['epochen']}")

    bewertet = ~np.isnan(proba).any(axis=1)          # Holdout: nur Testmenge
    pred = (proba >= SCHWELLE).astype(int)
    yt, yp, yb = y[bewertet], pred[bewertet], baseline[bewertet]
    ohne_us = bewertet & ~REINE_USER_STUDY

    ergebnisse.append({
        "seed": seed,
        "modell": metriken(yt, yp),
        "baseline": metriken(yt, yb),
        "modell_ohne_reine_us": metriken(y[ohne_us], pred[ohne_us]),
        "proba_sha256": hashlib.sha256(np.round(proba[bewertet], 12).tobytes()).hexdigest(),
    })

    for i in np.flatnonzero(bewertet):
        zeile = {"paper_id": paper_ids[i], "wiederholung": wdh, "seed": seed,
                 "reine_user_study": bool(REINE_USER_STUDY[i])}
        for j, spalte in enumerate(ML_COLUMNS):
            zeile[f"{spalte}_true"] = int(y[i, j])
            zeile[f"{spalte}_proba"] = round(float(proba[i, j]), 6)
            zeile[f"{spalte}_pred"] = int(pred[i, j])
        oof_zeilen.append(zeile)

    print(f"Wiederholung {wdh} (Seed {seed}): Makro-F1 {ergebnisse[-1]['modell']['macro_f1']:.3f}\n")

# =============================================================================
# ZUSAMMENFASSUNG UND SPEICHERUNG
# =============================================================================
def mittel_std(schluessel: str, wert: str):
    werte = np.array([e[schluessel][wert] for e in ergebnisse])
    return werte.mean(axis=0), werte.std(axis=0)


f1_m, f1_s = mittel_std("modell", "f1")
p_m, _ = mittel_std("modell", "precision")
r_m, _ = mittel_std("modell", "recall")
bf1_m, _ = mittel_std("baseline", "f1")
uf1_m, uf1_s = mittel_std("modell_ohne_reine_us", "f1")

pro_ziel = pd.DataFrame({
    "ziel": ML_COLUMNS,
    "support": ergebnisse[0]["modell"]["support"],
    "precision": p_m, "recall": r_m, "f1": f1_m, "f1_std": f1_s,
    "f1_mehrheitsklasse": bf1_m,
    "f1_ohne_reine_user_studies": uf1_m,
})
pro_ziel.to_csv(RUN_DIR / "metriken_pro_ziel.csv", index=False)
pd.DataFrame(oof_zeilen).to_csv(RUN_DIR / "vorhersagen.csv", index=False)
pd.DataFrame(verlaeufe).to_csv(RUN_DIR / "training_verlauf.csv", index=False)

gesamt = {}
for schluessel in ["modell", "baseline", "modell_ohne_reine_us"]:
    for wert in ["macro_f1", "hamming_loss", "subset_accuracy", "jaccard_samples"]:
        m, s = mittel_std(schluessel, wert)
        gesamt[f"{schluessel}.{wert}"] = (float(m), float(s))

def _version(paket: str) -> str:
    try:
        return paket_version(paket)
    except Exception:
        return "unbekannt"


versionen = {p: _version(p) for p in ["scikit-learn", "numpy", "iterative-stratification"]}
konfiguration = {
    "klassifikator": classifier_name,
    "validierung": VALIDIERUNGS_METHODE, "n_folds": N_FOLDS, "seeds": SEEDS,
    "holdout_testanteil": HOLDOUT_TESTANTEIL,
    "mlp_params": {**MLP_PARAMS, "hidden_layer_sizes": list(MLP_PARAMS["hidden_layer_sizes"])},
    "early_stopping": {"kriterium": "Validierungs-Loss (BCE)", "max_epochen": MAX_EPOCHEN,
                       "geduld": GEDULD, "min_verbesserung": MIN_VERBESSERUNG, "val_anteil": VAL_ANTEIL},
    "schwelle": SCHWELLE,
    "fusion_manifest": fusion_manifest,
    "versionen": versionen,
    "proba_sha256_je_seed": {e["seed"]: e["proba_sha256"] for e in ergebnisse},
    "ergebnis": gesamt,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(konfiguration, indent=2, default=str), encoding="utf-8")
save_run_config(extra={"ml_run": str(RUN_DIR), **konfiguration})

zeilen = [f"{k:<38} {m:.4f} +/- {s:.4f}" for k, (m, s) in gesamt.items()]
report = (
    f"Klassifikator: {classifier_name} | Evaluierung: {VALIDIERUNGS_METHODE} | Seeds: {SEEDS}\n"
    f"Daten: {DATEN_QUELLE} | {MODELL_NAME} | {STRATEGIE_NAME} | Pooling: {POOLING_STRATEGIE} | X {X.shape}\n"
    f"MLP: {MLP_PARAMS} | Early Stopping: Val-Loss, Geduld {GEDULD}\n\n"
    "Kennzahlen (Mittel +/- Standardabweichung ueber Wiederholungen):\n" + "\n".join(zeilen) + "\n\n"
    "Pro Ziel:\n" + pro_ziel.to_string(index=False, float_format=lambda v: f"{v:.3f}") + "\n"
)
(RUN_DIR / "REPORT.txt").write_text(report, encoding="utf-8")
print(report)
print(f"Gespeichert unter: {RUN_DIR}")

Daten: X (654, 1024), y (654, 14) | Evaluierung: cross_validation, Seeds [42, 43, 44, 45, 46]
Run-Ordner: /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/Predictions/Scikit_MLP_CrossVal/pdf_raw/max/mxbai-embed-large/recursive_1200c_200o_norefs/Run_1
  Wdh 1 Fold 1: beste Epoche 1 von 12
  Wdh 1 Fold 2: beste Epoche 1 von 12
  Wdh 1 Fold 3: beste Epoche 2 von 13
  Wdh 1 Fold 4: beste Epoche 2 von 13
  Wdh 1 Fold 5: beste Epoche 2 von 13
Wiederholung 1 (Seed 42): Makro-F1 0.501

  Wdh 2 Fold 1: beste Epoche 2 von 13
  Wdh 2 Fold 2: beste Epoche 1 von 12
  Wdh 2 Fold 3: beste Epoche 1 von 12
  Wdh 2 Fold 4: beste Epoche 2 von 13
  Wdh 2 Fold 5: beste Epoche 1 von 12
Wiederholung 2 (Seed 43): Makro-F1 0.505

  Wdh 3 Fold 1: beste Epoche 1 von 12
  Wdh 3 Fold 2: beste Epoche 1 von 12
  Wdh 3 Fold 3: beste Epoche 1 von 12
  Wdh 3 Fold 4: beste Epoche 1 von 12
  Wdh 3 Fold 5: beste Epoche 1 von 12
Wiederholung 3 (Seed 44): Makro-F1 0.501

  Wdh 4 Fold 1: beste Epoche 1 von 12
  Wdh 4 Fold 2

In [7]:
"""
Segment 6: Multi-Label-Klassifikation mit PyTorch (MLP, BCEWithLogitsLoss).

Gleiches Evaluationsprotokoll wie Segment 5 (scikit-learn), damit beide Modelle
direkt vergleichbar sind: dieselben Seeds, dieselben Fold-Aufteilungen, dieselbe
innere Validierungsmenge, dieselben Metriken, Basislinie und Sensitivitaetsanalyse.

Unterschiede zum scikit-learn-MLP (bewusste Entwurfsentscheidungen):
- Dropout (0.3) nach jeder verborgenen Schicht
- optional pos_weight je Label gegen das Klassenungleichgewicht (NUTZE_GEWICHTUNG);
  das verschiebt die Wahrscheinlichkeiten nach oben (hoeherer Recall, geringere Precision)

Varianten (VARIANTE), gleicher Code, unterschiedliche Einstellungen:
- "standard": pos_weight ungedeckelt, keine L2-Regularisierung, feste Lernrate
- "balanced": pos_weight auf 25 gedeckelt, L2 (weight_decay 1e-4), ReduceLROnPlateau
  (Faktor 0.5, Geduld 3) auf dem Validierungs-Loss. Der Deckel greift bei diesem
  Datensatz nur fuer Literature Review (ungedeckelt ca. 53).

Early Stopping auf dem Validierungs-Loss einer inneren, stratifizierten
Validierungsmenge; die Gewichte der besten Epoche werden wiederhergestellt.
Die Testmenge wird erst nach dem Training einmalig vorhergesagt.
"""

import copy
from importlib.metadata import version as paket_version

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
from sklearn.metrics import accuracy_score, hamming_loss, jaccard_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# =============================================================================
# KONFIGURATION
# =============================================================================
VALIDIERUNGS_METHODE = "cross_validation"   # "cross_validation" | "holdout"
N_FOLDS = 5
N_WIEDERHOLUNGEN = 5                        # Seeds: GLOBAL_SEED, GLOBAL_SEED + 1, ...
HOLDOUT_TESTANTEIL = 0.2
NUTZE_GEWICHTUNG = True                     # pos_weight = negative / positive Faelle je Label
VARIANTE = "standard"                       # "standard" | "balanced"

VARIANTEN = {
    "standard": {"pos_weight_max": None, "weight_decay": 0.0, "lr_scheduler": None},
    "balanced": {"pos_weight_max": 25.0, "weight_decay": 1e-4,
                 "lr_scheduler": {"factor": 0.5, "patience": 3}},
}
EINSTELLUNGEN = VARIANTEN[VARIANTE]

NETZ_PARAMS = {"verborgene_schichten": [512, 256], "dropout": 0.3}
LERNRATE = 1e-3             # Startwert; bei "balanced" vom Scheduler reduziert
BATCH_SIZE = 128
MAX_EPOCHEN = 200
GEDULD = 10                 # Epochen ohne Verbesserung des Validierungs-Loss
MIN_VERBESSERUNG = 1e-4
VAL_ANTEIL = 0.1            # wie Segment 5
SCHWELLE = 0.5

classifier_name = f"PyTorch_BCE_MLP_{VARIANTE}" + ("" if NUTZE_GEWICHTUNG else "_ohne_posweight")
# Feste, aus GLOBAL_SEED abgeleitete Seeds: jede Ausfuehrung nutzt dieselben Splits und Initialisierungen
SEEDS = [GLOBAL_SEED + i for i in range(N_WIEDERHOLUNGEN)]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# DATEN
# =============================================================================
X = np.load(CURRENT_FUSION_PATH / "X_data.npy")
y = np.load(CURRENT_FUSION_PATH / "y_data.npy")
ML_COLUMNS = json.loads((CURRENT_FUSION_PATH / "y_columns.json").read_text(encoding="utf-8"))
paper_ids = json.loads((CURRENT_FUSION_PATH / "paper_ids.json").read_text(encoding="utf-8"))
fusion_manifest = json.loads((CURRENT_FUSION_PATH / "fusion_manifest.json").read_text(encoding="utf-8"))

if not (len(X) == len(y) == len(paper_ids)) or y.shape[1] != len(ML_COLUMNS):
    raise ValueError("X, y, paper_ids und y_columns passen nicht zusammen. Segment 4 erneut ausfuehren.")

typ_roh = master_label_raw_df.set_index("paper_id")["paper_type"].astype(str).str.strip()
REINE_USER_STUDY = np.array([typ_roh.get(p) == "User Study" for p in paper_ids])

# =============================================================================
# RUN-ORDNER
# =============================================================================
base_dir = PREDICTIONS_ML_ROOT / classifier_name / DATEN_QUELLE / POOLING_STRATEGIE / MODELL_NAME / STRATEGIE_NAME
base_dir.mkdir(parents=True, exist_ok=True)
run_nummern = [int(d.name.split("_")[1]) for d in base_dir.glob("Run_*") if d.name.split("_")[1].isdigit()]
RUN_DIR = base_dir / f"Run_{max(run_nummern, default=0) + 1}"
RUN_DIR.mkdir()

print(f"Device: {DEVICE} | X {X.shape}, y {y.shape} | {VALIDIERUNGS_METHODE}, Seeds {SEEDS}")
print(f"Run-Ordner: {RUN_DIR}")


# =============================================================================
# MODELL UND TRAINING
# =============================================================================
class MultiLabelMLP(nn.Module):
    """Vollvernetztes MLP; Ausgabe sind Logits (Sigmoid steckt in BCEWithLogitsLoss)."""

    def __init__(self, input_dim: int, output_dim: int, verborgene_schichten: list, dropout: float):
        super().__init__()
        schichten, dim = [], input_dim
        for breite in verborgene_schichten:
            schichten += [nn.Linear(dim, breite), nn.ReLU(), nn.Dropout(dropout)]
            dim = breite
        schichten.append(nn.Linear(dim, output_dim))
        self.network = nn.Sequential(*schichten)

    def forward(self, x):
        return self.network(x)


def tensoren(X_teil: np.ndarray, y_teil: np.ndarray) -> TensorDataset:
    return TensorDataset(torch.tensor(X_teil, dtype=torch.float32), torch.tensor(y_teil, dtype=torch.float32))


def mittlerer_loss(model, loader, kriterium) -> float:
    model.eval()
    summe, anzahl = 0.0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            summe += kriterium(model(bx), by).item() * len(bx)
            anzahl += len(bx)
    return summe / anzahl


def trainiere_mlp(X_tr: np.ndarray, y_tr: np.ndarray, seed: int):
    """
    Trainiert ein MLP mit Early Stopping auf dem Validierungs-Loss.
    Rueckgabe: (Modell mit besten Gewichten, Scaler, Verlauf).
    """
    innen_tr, innen_val = next(MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=VAL_ANTEIL, random_state=seed).split(X_tr, y_tr))

    scaler = StandardScaler().fit(X_tr[innen_tr])
    Xa, Xv = scaler.transform(X_tr[innen_tr]), scaler.transform(X_tr[innen_val])
    ya, yv = y_tr[innen_tr], y_tr[innen_val]

    # Seed der Wiederholung (aus GLOBAL_SEED abgeleitet, bei jeder Ausfuehrung gleich) vor
    # jedem Training neu setzen: Initialisierung, Dropout und Batch-Reihenfolge sind damit
    # identisch reproduzierbar und haengen nicht davon ab, welche Folds vorher gelaufen sind.
    torch.manual_seed(seed)
    batch_generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(tensoren(Xa, ya), batch_size=BATCH_SIZE, shuffle=True, generator=batch_generator)
    val_loader = DataLoader(tensoren(Xv, yv), batch_size=BATCH_SIZE, shuffle=False)

    pos_weight = None
    if NUTZE_GEWICHTUNG:
        positiv = np.clip(ya.sum(axis=0), 1, None)
        gewichte = (len(ya) - positiv) / positiv
        if EINSTELLUNGEN["pos_weight_max"] is not None:
            gewichte = np.minimum(gewichte, EINSTELLUNGEN["pos_weight_max"])
        pos_weight = torch.tensor(gewichte, dtype=torch.float32, device=DEVICE)
    kriterium = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model = MultiLabelMLP(X_tr.shape[1], y_tr.shape[1], **NETZ_PARAMS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LERNRATE, weight_decay=EINSTELLUNGEN["weight_decay"])
    scheduler = None
    if EINSTELLUNGEN["lr_scheduler"]:
        # Reagiert ausschliesslich auf den inneren Validierungs-Loss, nie auf den Test-Fold
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", **EINSTELLUNGEN["lr_scheduler"])

    verlauf = {"train_loss": [], "val_loss": [], "lr": []}
    bester_loss, beste_epoche, ohne_verbesserung = np.inf, 0, 0
    beste_gewichte = copy.deepcopy(model.state_dict())

    for epoche in range(MAX_EPOCHEN):
        model.train()
        summe = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            loss = kriterium(model(bx), by)
            loss.backward()
            optimizer.step()
            summe += loss.item() * len(bx)
        verlauf["train_loss"].append(summe / len(Xa))
        val_loss = mittlerer_loss(model, val_loader, kriterium)
        verlauf["val_loss"].append(val_loss)
        verlauf["lr"].append(optimizer.param_groups[0]["lr"])
        if scheduler is not None:
            scheduler.step(val_loss)

        if val_loss < bester_loss - MIN_VERBESSERUNG:
            bester_loss, beste_epoche, ohne_verbesserung = val_loss, epoche, 0
            beste_gewichte = copy.deepcopy(model.state_dict())
        else:
            ohne_verbesserung += 1
            if ohne_verbesserung >= GEDULD:
                break

    model.load_state_dict(beste_gewichte)
    model.eval()
    verlauf["beste_epoche"] = beste_epoche
    verlauf["epochen"] = epoche + 1
    return model, scaler, verlauf


def vorhersage_proba(model, scaler, X_test: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        logits = model(torch.tensor(scaler.transform(X_test), dtype=torch.float32, device=DEVICE))
        return torch.sigmoid(logits).cpu().numpy()


def splits(seed: int):
    if VALIDIERUNGS_METHODE == "cross_validation":
        return MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(X, y)
    return MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=HOLDOUT_TESTANTEIL, random_state=seed).split(X, y)


def plotte_verlauf(verlauf: dict, titel: str, pfad: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(verlauf["train_loss"], label="Trainings-Loss")
    ax.plot(verlauf["val_loss"], label="Validierungs-Loss")
    ax.axvline(verlauf["beste_epoche"], color="red", linestyle="--",
               label=f"Beste Epoche ({verlauf['beste_epoche']})")
    ax.set(title=titel, xlabel="Epoche",
           ylabel="BCE" + (" (mit pos_weight)" if NUTZE_GEWICHTUNG else ""))
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)
    fig.tight_layout()
    fig.savefig(pfad, dpi=150)
    plt.close(fig)


# =============================================================================
# EVALUIERUNG (identisch zu Segment 5)
# =============================================================================
def metriken(y_wahr: np.ndarray, y_pred: np.ndarray) -> dict:
    p, r, f, s = precision_recall_fscore_support(y_wahr, y_pred, average=None, zero_division=0)
    return {
        "precision": p, "recall": r, "f1": f, "support": s,
        "macro_f1": float(f.mean()),
        "hamming_loss": hamming_loss(y_wahr, y_pred),
        "subset_accuracy": accuracy_score(y_wahr, y_pred),
        "jaccard_samples": jaccard_score(y_wahr, y_pred, average="samples", zero_division=0),
    }


ergebnisse, oof_zeilen, verlaeufe = [], [], []

for wdh, seed in enumerate(SEEDS, start=1):
    proba = np.full(y.shape, np.nan)
    baseline = np.full(y.shape, -1, dtype=int)

    for fold, (train_idx, test_idx) in enumerate(splits(seed), start=1):
        modell, scaler, verlauf = trainiere_mlp(X[train_idx], y[train_idx], seed)
        proba[test_idx] = vorhersage_proba(modell, scaler, X[test_idx])
        baseline[test_idx] = (y[train_idx].mean(axis=0) >= 0.5).astype(int)

        verlaeufe.append({"wiederholung": wdh, "seed": seed, "fold": fold,
                          "beste_epoche": verlauf["beste_epoche"], "epochen": verlauf["epochen"],
                          "bester_val_loss": min(verlauf["val_loss"]),
                          "lr_am_ende": verlauf["lr"][-1]})
        if wdh == 1:
            plotte_verlauf(verlauf, f"Wiederholung 1, Fold {fold}", RUN_DIR / f"lernkurve_fold-{fold}.png")
        print(f"  Wdh {wdh} Fold {fold}: beste Epoche {verlauf['beste_epoche']} von {verlauf['epochen']}")

    bewertet = ~np.isnan(proba).any(axis=1)
    pred = (proba >= SCHWELLE).astype(int)
    yt, yp, yb = y[bewertet], pred[bewertet], baseline[bewertet]
    ohne_us = bewertet & ~REINE_USER_STUDY

    ergebnisse.append({
        "seed": seed,
        "modell": metriken(yt, yp),
        "baseline": metriken(yt, yb),
        "modell_ohne_reine_us": metriken(y[ohne_us], pred[ohne_us]),
        "proba_sha256": hashlib.sha256(np.round(proba[bewertet], 6).tobytes()).hexdigest(),
    })

    for i in np.flatnonzero(bewertet):
        zeile = {"paper_id": paper_ids[i], "wiederholung": wdh, "seed": seed,
                 "reine_user_study": bool(REINE_USER_STUDY[i])}
        for j, spalte in enumerate(ML_COLUMNS):
            zeile[f"{spalte}_true"] = int(y[i, j])
            zeile[f"{spalte}_proba"] = round(float(proba[i, j]), 6)
            zeile[f"{spalte}_pred"] = int(pred[i, j])
        oof_zeilen.append(zeile)

    print(f"Wiederholung {wdh} (Seed {seed}): Makro-F1 {ergebnisse[-1]['modell']['macro_f1']:.3f}\n")

# =============================================================================
# ZUSAMMENFASSUNG UND SPEICHERUNG
# =============================================================================
def mittel_std(schluessel: str, wert: str):
    werte = np.array([e[schluessel][wert] for e in ergebnisse])
    return werte.mean(axis=0), werte.std(axis=0)


f1_m, f1_s = mittel_std("modell", "f1")
p_m, _ = mittel_std("modell", "precision")
r_m, _ = mittel_std("modell", "recall")
bf1_m, _ = mittel_std("baseline", "f1")
uf1_m, _ = mittel_std("modell_ohne_reine_us", "f1")

pro_ziel = pd.DataFrame({
    "ziel": ML_COLUMNS,
    "support": ergebnisse[0]["modell"]["support"],
    "precision": p_m, "recall": r_m, "f1": f1_m, "f1_std": f1_s,
    "f1_mehrheitsklasse": bf1_m,
    "f1_ohne_reine_user_studies": uf1_m,
})
pro_ziel.to_csv(RUN_DIR / "metriken_pro_ziel.csv", index=False)
pd.DataFrame(oof_zeilen).to_csv(RUN_DIR / "vorhersagen.csv", index=False)
pd.DataFrame(verlaeufe).to_csv(RUN_DIR / "training_verlauf.csv", index=False)

gesamt = {}
for schluessel in ["modell", "baseline", "modell_ohne_reine_us"]:
    for wert in ["macro_f1", "hamming_loss", "subset_accuracy", "jaccard_samples"]:
        m, s = mittel_std(schluessel, wert)
        gesamt[f"{schluessel}.{wert}"] = (float(m), float(s))


def _version(paket: str) -> str:
    try:
        return paket_version(paket)
    except Exception:
        return "unbekannt"


konfiguration = {
    "klassifikator": classifier_name,
    "validierung": VALIDIERUNGS_METHODE, "n_folds": N_FOLDS, "seeds": SEEDS,
    "holdout_testanteil": HOLDOUT_TESTANTEIL,
    "netz": NETZ_PARAMS, "lernrate": LERNRATE, "batch_size": BATCH_SIZE,
    "pos_weight": NUTZE_GEWICHTUNG,
    "variante": VARIANTE, "einstellungen": EINSTELLUNGEN,
    "early_stopping": {"kriterium": "Validierungs-Loss (BCE)", "max_epochen": MAX_EPOCHEN,
                       "geduld": GEDULD, "min_verbesserung": MIN_VERBESSERUNG, "val_anteil": VAL_ANTEIL},
    "schwelle": SCHWELLE,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "versionen": {
        "torch": torch.__version__, "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version() if DEVICE.type == "cuda" else None,
        **{p: _version(p) for p in ["scikit-learn", "numpy", "iterative-stratification"]},
    },
    "deterministische_algorithmen": torch.are_deterministic_algorithms_enabled(),
    "fusion_manifest": fusion_manifest,
    "proba_sha256_je_seed": {e["seed"]: e["proba_sha256"] for e in ergebnisse},
    "ergebnis": gesamt,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(konfiguration, indent=2, default=str), encoding="utf-8")
save_run_config(extra={"ml_run": str(RUN_DIR), **konfiguration})

zeilen = [f"{k:<38} {m:.4f} +/- {s:.4f}" for k, (m, s) in gesamt.items()]
report = (
    f"Klassifikator: {classifier_name} | Evaluierung: {VALIDIERUNGS_METHODE} | Seeds: {SEEDS} | Device: {DEVICE}\n"
    f"Daten: {DATEN_QUELLE} | {MODELL_NAME} | {STRATEGIE_NAME} | Pooling: {POOLING_STRATEGIE} | X {X.shape}\n"
    f"Variante: {VARIANTE} {EINSTELLUNGEN}\n"
    f"Netz: {NETZ_PARAMS} | Lernrate {LERNRATE} | Batch {BATCH_SIZE} | pos_weight: {NUTZE_GEWICHTUNG} | "
    f"Early Stopping: Val-Loss, Geduld {GEDULD}\n\n"
    "Kennzahlen (Mittel +/- Standardabweichung ueber Wiederholungen):\n" + "\n".join(zeilen) + "\n\n"
    "Pro Ziel:\n" + pro_ziel.to_string(index=False, float_format=lambda v: f"{v:.3f}") + "\n"
)
(RUN_DIR / "REPORT.txt").write_text(report, encoding="utf-8")
print(report)
print(f"Gespeichert unter: {RUN_DIR}")

Device: cpu | X (654, 1024), y (654, 14) | cross_validation, Seeds [42, 43, 44, 45, 46]
Run-Ordner: /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/Predictions/PyTorch_BCE_MLP_standard/pdf_raw/max/mxbai-embed-large/recursive_1200c_200o_norefs/Run_1
  Wdh 1 Fold 1: beste Epoche 1 von 12
  Wdh 1 Fold 2: beste Epoche 2 von 13
  Wdh 1 Fold 3: beste Epoche 2 von 13
  Wdh 1 Fold 4: beste Epoche 2 von 13
  Wdh 1 Fold 5: beste Epoche 2 von 13
Wiederholung 1 (Seed 42): Makro-F1 0.562

  Wdh 2 Fold 1: beste Epoche 4 von 15
  Wdh 2 Fold 2: beste Epoche 3 von 14
  Wdh 2 Fold 3: beste Epoche 6 von 17
  Wdh 2 Fold 4: beste Epoche 2 von 13
  Wdh 2 Fold 5: beste Epoche 1 von 12
Wiederholung 2 (Seed 43): Makro-F1 0.568

  Wdh 3 Fold 1: beste Epoche 2 von 13
  Wdh 3 Fold 2: beste Epoche 2 von 13
  Wdh 3 Fold 3: beste Epoche 2 von 13
  Wdh 3 Fold 4: beste Epoche 1 von 12
  Wdh 3 Fold 5: beste Epoche 3 von 14
Wiederholung 3 (Seed 44): Makro-F1 0.566

  Wdh 4 Fold 1: beste Epoche 5 von 16
  Wdh 4 Fold 2:

In [8]:
"""
Segment 6: Multi-Label-Klassifikation mit PyTorch (MLP, BCEWithLogitsLoss).

Gleiches Evaluationsprotokoll wie Segment 5 (scikit-learn), damit beide Modelle
direkt vergleichbar sind: dieselben Seeds, dieselben Fold-Aufteilungen, dieselbe
innere Validierungsmenge, dieselben Metriken, Basislinie und Sensitivitaetsanalyse.

Unterschiede zum scikit-learn-MLP (bewusste Entwurfsentscheidungen):
- Dropout (0.3) nach jeder verborgenen Schicht
- optional pos_weight je Label gegen das Klassenungleichgewicht (NUTZE_GEWICHTUNG);
  das verschiebt die Wahrscheinlichkeiten nach oben (hoeherer Recall, geringere Precision)

Varianten (VARIANTE), gleicher Code, unterschiedliche Einstellungen:
- "standard": pos_weight ungedeckelt, keine L2-Regularisierung, feste Lernrate
- "balanced": pos_weight auf 25 gedeckelt, L2 (weight_decay 1e-4), ReduceLROnPlateau
  (Faktor 0.5, Geduld 3) auf dem Validierungs-Loss. Der Deckel greift bei diesem
  Datensatz nur fuer Literature Review (ungedeckelt ca. 53).

Early Stopping auf dem Validierungs-Loss einer inneren, stratifizierten
Validierungsmenge; die Gewichte der besten Epoche werden wiederhergestellt.
Die Testmenge wird erst nach dem Training einmalig vorhergesagt.
"""

import copy
from importlib.metadata import version as paket_version

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
from sklearn.metrics import accuracy_score, hamming_loss, jaccard_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# =============================================================================
# KONFIGURATION
# =============================================================================
VALIDIERUNGS_METHODE = "cross_validation"   # "cross_validation" | "holdout"
N_FOLDS = 5
N_WIEDERHOLUNGEN = 5                        # Seeds: GLOBAL_SEED, GLOBAL_SEED + 1, ...
HOLDOUT_TESTANTEIL = 0.2
NUTZE_GEWICHTUNG = True                     # pos_weight = negative / positive Faelle je Label
VARIANTE = "balanced"                       # "standard" | "balanced"

VARIANTEN = {
    "standard": {"pos_weight_max": None, "weight_decay": 0.0, "lr_scheduler": None},
    "balanced": {"pos_weight_max": 25.0, "weight_decay": 1e-4,
                 "lr_scheduler": {"factor": 0.5, "patience": 3}},
}
EINSTELLUNGEN = VARIANTEN[VARIANTE]

NETZ_PARAMS = {"verborgene_schichten": [512, 256], "dropout": 0.3}
LERNRATE = 1e-3             # Startwert; bei "balanced" vom Scheduler reduziert
BATCH_SIZE = 128
MAX_EPOCHEN = 200
GEDULD = 10                 # Epochen ohne Verbesserung des Validierungs-Loss
MIN_VERBESSERUNG = 1e-4
VAL_ANTEIL = 0.1            # wie Segment 5
SCHWELLE = 0.5

classifier_name = f"PyTorch_BCE_MLP_{VARIANTE}" + ("" if NUTZE_GEWICHTUNG else "_ohne_posweight")
# Feste, aus GLOBAL_SEED abgeleitete Seeds: jede Ausfuehrung nutzt dieselben Splits und Initialisierungen
SEEDS = [GLOBAL_SEED + i for i in range(N_WIEDERHOLUNGEN)]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# DATEN
# =============================================================================
X = np.load(CURRENT_FUSION_PATH / "X_data.npy")
y = np.load(CURRENT_FUSION_PATH / "y_data.npy")
ML_COLUMNS = json.loads((CURRENT_FUSION_PATH / "y_columns.json").read_text(encoding="utf-8"))
paper_ids = json.loads((CURRENT_FUSION_PATH / "paper_ids.json").read_text(encoding="utf-8"))
fusion_manifest = json.loads((CURRENT_FUSION_PATH / "fusion_manifest.json").read_text(encoding="utf-8"))

if not (len(X) == len(y) == len(paper_ids)) or y.shape[1] != len(ML_COLUMNS):
    raise ValueError("X, y, paper_ids und y_columns passen nicht zusammen. Segment 4 erneut ausfuehren.")

typ_roh = master_label_raw_df.set_index("paper_id")["paper_type"].astype(str).str.strip()
REINE_USER_STUDY = np.array([typ_roh.get(p) == "User Study" for p in paper_ids])

# =============================================================================
# RUN-ORDNER
# =============================================================================
base_dir = PREDICTIONS_ML_ROOT / classifier_name / DATEN_QUELLE / POOLING_STRATEGIE / MODELL_NAME / STRATEGIE_NAME
base_dir.mkdir(parents=True, exist_ok=True)
run_nummern = [int(d.name.split("_")[1]) for d in base_dir.glob("Run_*") if d.name.split("_")[1].isdigit()]
RUN_DIR = base_dir / f"Run_{max(run_nummern, default=0) + 1}"
RUN_DIR.mkdir()

print(f"Device: {DEVICE} | X {X.shape}, y {y.shape} | {VALIDIERUNGS_METHODE}, Seeds {SEEDS}")
print(f"Run-Ordner: {RUN_DIR}")


# =============================================================================
# MODELL UND TRAINING
# =============================================================================
class MultiLabelMLP(nn.Module):
    """Vollvernetztes MLP; Ausgabe sind Logits (Sigmoid steckt in BCEWithLogitsLoss)."""

    def __init__(self, input_dim: int, output_dim: int, verborgene_schichten: list, dropout: float):
        super().__init__()
        schichten, dim = [], input_dim
        for breite in verborgene_schichten:
            schichten += [nn.Linear(dim, breite), nn.ReLU(), nn.Dropout(dropout)]
            dim = breite
        schichten.append(nn.Linear(dim, output_dim))
        self.network = nn.Sequential(*schichten)

    def forward(self, x):
        return self.network(x)


def tensoren(X_teil: np.ndarray, y_teil: np.ndarray) -> TensorDataset:
    return TensorDataset(torch.tensor(X_teil, dtype=torch.float32), torch.tensor(y_teil, dtype=torch.float32))


def mittlerer_loss(model, loader, kriterium) -> float:
    model.eval()
    summe, anzahl = 0.0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            summe += kriterium(model(bx), by).item() * len(bx)
            anzahl += len(bx)
    return summe / anzahl


def trainiere_mlp(X_tr: np.ndarray, y_tr: np.ndarray, seed: int):
    """
    Trainiert ein MLP mit Early Stopping auf dem Validierungs-Loss.
    Rueckgabe: (Modell mit besten Gewichten, Scaler, Verlauf).
    """
    innen_tr, innen_val = next(MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=VAL_ANTEIL, random_state=seed).split(X_tr, y_tr))

    scaler = StandardScaler().fit(X_tr[innen_tr])
    Xa, Xv = scaler.transform(X_tr[innen_tr]), scaler.transform(X_tr[innen_val])
    ya, yv = y_tr[innen_tr], y_tr[innen_val]

    # Seed der Wiederholung (aus GLOBAL_SEED abgeleitet, bei jeder Ausfuehrung gleich) vor
    # jedem Training neu setzen: Initialisierung, Dropout und Batch-Reihenfolge sind damit
    # identisch reproduzierbar und haengen nicht davon ab, welche Folds vorher gelaufen sind.
    torch.manual_seed(seed)
    batch_generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(tensoren(Xa, ya), batch_size=BATCH_SIZE, shuffle=True, generator=batch_generator)
    val_loader = DataLoader(tensoren(Xv, yv), batch_size=BATCH_SIZE, shuffle=False)

    pos_weight = None
    if NUTZE_GEWICHTUNG:
        positiv = np.clip(ya.sum(axis=0), 1, None)
        gewichte = (len(ya) - positiv) / positiv
        if EINSTELLUNGEN["pos_weight_max"] is not None:
            gewichte = np.minimum(gewichte, EINSTELLUNGEN["pos_weight_max"])
        pos_weight = torch.tensor(gewichte, dtype=torch.float32, device=DEVICE)
    kriterium = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model = MultiLabelMLP(X_tr.shape[1], y_tr.shape[1], **NETZ_PARAMS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LERNRATE, weight_decay=EINSTELLUNGEN["weight_decay"])
    scheduler = None
    if EINSTELLUNGEN["lr_scheduler"]:
        # Reagiert ausschliesslich auf den inneren Validierungs-Loss, nie auf den Test-Fold
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", **EINSTELLUNGEN["lr_scheduler"])

    verlauf = {"train_loss": [], "val_loss": [], "lr": []}
    bester_loss, beste_epoche, ohne_verbesserung = np.inf, 0, 0
    beste_gewichte = copy.deepcopy(model.state_dict())

    for epoche in range(MAX_EPOCHEN):
        model.train()
        summe = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            loss = kriterium(model(bx), by)
            loss.backward()
            optimizer.step()
            summe += loss.item() * len(bx)
        verlauf["train_loss"].append(summe / len(Xa))
        val_loss = mittlerer_loss(model, val_loader, kriterium)
        verlauf["val_loss"].append(val_loss)
        verlauf["lr"].append(optimizer.param_groups[0]["lr"])
        if scheduler is not None:
            scheduler.step(val_loss)

        if val_loss < bester_loss - MIN_VERBESSERUNG:
            bester_loss, beste_epoche, ohne_verbesserung = val_loss, epoche, 0
            beste_gewichte = copy.deepcopy(model.state_dict())
        else:
            ohne_verbesserung += 1
            if ohne_verbesserung >= GEDULD:
                break

    model.load_state_dict(beste_gewichte)
    model.eval()
    verlauf["beste_epoche"] = beste_epoche
    verlauf["epochen"] = epoche + 1
    return model, scaler, verlauf


def vorhersage_proba(model, scaler, X_test: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        logits = model(torch.tensor(scaler.transform(X_test), dtype=torch.float32, device=DEVICE))
        return torch.sigmoid(logits).cpu().numpy()


def splits(seed: int):
    if VALIDIERUNGS_METHODE == "cross_validation":
        return MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(X, y)
    return MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=HOLDOUT_TESTANTEIL, random_state=seed).split(X, y)


def plotte_verlauf(verlauf: dict, titel: str, pfad: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(verlauf["train_loss"], label="Trainings-Loss")
    ax.plot(verlauf["val_loss"], label="Validierungs-Loss")
    ax.axvline(verlauf["beste_epoche"], color="red", linestyle="--",
               label=f"Beste Epoche ({verlauf['beste_epoche']})")
    ax.set(title=titel, xlabel="Epoche",
           ylabel="BCE" + (" (mit pos_weight)" if NUTZE_GEWICHTUNG else ""))
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)
    fig.tight_layout()
    fig.savefig(pfad, dpi=150)
    plt.close(fig)


# =============================================================================
# EVALUIERUNG (identisch zu Segment 5)
# =============================================================================
def metriken(y_wahr: np.ndarray, y_pred: np.ndarray) -> dict:
    p, r, f, s = precision_recall_fscore_support(y_wahr, y_pred, average=None, zero_division=0)
    return {
        "precision": p, "recall": r, "f1": f, "support": s,
        "macro_f1": float(f.mean()),
        "hamming_loss": hamming_loss(y_wahr, y_pred),
        "subset_accuracy": accuracy_score(y_wahr, y_pred),
        "jaccard_samples": jaccard_score(y_wahr, y_pred, average="samples", zero_division=0),
    }


ergebnisse, oof_zeilen, verlaeufe = [], [], []

for wdh, seed in enumerate(SEEDS, start=1):
    proba = np.full(y.shape, np.nan)
    baseline = np.full(y.shape, -1, dtype=int)

    for fold, (train_idx, test_idx) in enumerate(splits(seed), start=1):
        modell, scaler, verlauf = trainiere_mlp(X[train_idx], y[train_idx], seed)
        proba[test_idx] = vorhersage_proba(modell, scaler, X[test_idx])
        baseline[test_idx] = (y[train_idx].mean(axis=0) >= 0.5).astype(int)

        verlaeufe.append({"wiederholung": wdh, "seed": seed, "fold": fold,
                          "beste_epoche": verlauf["beste_epoche"], "epochen": verlauf["epochen"],
                          "bester_val_loss": min(verlauf["val_loss"]),
                          "lr_am_ende": verlauf["lr"][-1]})
        if wdh == 1:
            plotte_verlauf(verlauf, f"Wiederholung 1, Fold {fold}", RUN_DIR / f"lernkurve_fold-{fold}.png")
        print(f"  Wdh {wdh} Fold {fold}: beste Epoche {verlauf['beste_epoche']} von {verlauf['epochen']}")

    bewertet = ~np.isnan(proba).any(axis=1)
    pred = (proba >= SCHWELLE).astype(int)
    yt, yp, yb = y[bewertet], pred[bewertet], baseline[bewertet]
    ohne_us = bewertet & ~REINE_USER_STUDY

    ergebnisse.append({
        "seed": seed,
        "modell": metriken(yt, yp),
        "baseline": metriken(yt, yb),
        "modell_ohne_reine_us": metriken(y[ohne_us], pred[ohne_us]),
        "proba_sha256": hashlib.sha256(np.round(proba[bewertet], 6).tobytes()).hexdigest(),
    })

    for i in np.flatnonzero(bewertet):
        zeile = {"paper_id": paper_ids[i], "wiederholung": wdh, "seed": seed,
                 "reine_user_study": bool(REINE_USER_STUDY[i])}
        for j, spalte in enumerate(ML_COLUMNS):
            zeile[f"{spalte}_true"] = int(y[i, j])
            zeile[f"{spalte}_proba"] = round(float(proba[i, j]), 6)
            zeile[f"{spalte}_pred"] = int(pred[i, j])
        oof_zeilen.append(zeile)

    print(f"Wiederholung {wdh} (Seed {seed}): Makro-F1 {ergebnisse[-1]['modell']['macro_f1']:.3f}\n")

# =============================================================================
# ZUSAMMENFASSUNG UND SPEICHERUNG
# =============================================================================
def mittel_std(schluessel: str, wert: str):
    werte = np.array([e[schluessel][wert] for e in ergebnisse])
    return werte.mean(axis=0), werte.std(axis=0)


f1_m, f1_s = mittel_std("modell", "f1")
p_m, _ = mittel_std("modell", "precision")
r_m, _ = mittel_std("modell", "recall")
bf1_m, _ = mittel_std("baseline", "f1")
uf1_m, _ = mittel_std("modell_ohne_reine_us", "f1")

pro_ziel = pd.DataFrame({
    "ziel": ML_COLUMNS,
    "support": ergebnisse[0]["modell"]["support"],
    "precision": p_m, "recall": r_m, "f1": f1_m, "f1_std": f1_s,
    "f1_mehrheitsklasse": bf1_m,
    "f1_ohne_reine_user_studies": uf1_m,
})
pro_ziel.to_csv(RUN_DIR / "metriken_pro_ziel.csv", index=False)
pd.DataFrame(oof_zeilen).to_csv(RUN_DIR / "vorhersagen.csv", index=False)
pd.DataFrame(verlaeufe).to_csv(RUN_DIR / "training_verlauf.csv", index=False)

gesamt = {}
for schluessel in ["modell", "baseline", "modell_ohne_reine_us"]:
    for wert in ["macro_f1", "hamming_loss", "subset_accuracy", "jaccard_samples"]:
        m, s = mittel_std(schluessel, wert)
        gesamt[f"{schluessel}.{wert}"] = (float(m), float(s))


def _version(paket: str) -> str:
    try:
        return paket_version(paket)
    except Exception:
        return "unbekannt"


konfiguration = {
    "klassifikator": classifier_name,
    "validierung": VALIDIERUNGS_METHODE, "n_folds": N_FOLDS, "seeds": SEEDS,
    "holdout_testanteil": HOLDOUT_TESTANTEIL,
    "netz": NETZ_PARAMS, "lernrate": LERNRATE, "batch_size": BATCH_SIZE,
    "pos_weight": NUTZE_GEWICHTUNG,
    "variante": VARIANTE, "einstellungen": EINSTELLUNGEN,
    "early_stopping": {"kriterium": "Validierungs-Loss (BCE)", "max_epochen": MAX_EPOCHEN,
                       "geduld": GEDULD, "min_verbesserung": MIN_VERBESSERUNG, "val_anteil": VAL_ANTEIL},
    "schwelle": SCHWELLE,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "versionen": {
        "torch": torch.__version__, "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version() if DEVICE.type == "cuda" else None,
        **{p: _version(p) for p in ["scikit-learn", "numpy", "iterative-stratification"]},
    },
    "deterministische_algorithmen": torch.are_deterministic_algorithms_enabled(),
    "fusion_manifest": fusion_manifest,
    "proba_sha256_je_seed": {e["seed"]: e["proba_sha256"] for e in ergebnisse},
    "ergebnis": gesamt,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(konfiguration, indent=2, default=str), encoding="utf-8")
save_run_config(extra={"ml_run": str(RUN_DIR), **konfiguration})

zeilen = [f"{k:<38} {m:.4f} +/- {s:.4f}" for k, (m, s) in gesamt.items()]
report = (
    f"Klassifikator: {classifier_name} | Evaluierung: {VALIDIERUNGS_METHODE} | Seeds: {SEEDS} | Device: {DEVICE}\n"
    f"Daten: {DATEN_QUELLE} | {MODELL_NAME} | {STRATEGIE_NAME} | Pooling: {POOLING_STRATEGIE} | X {X.shape}\n"
    f"Variante: {VARIANTE} {EINSTELLUNGEN}\n"
    f"Netz: {NETZ_PARAMS} | Lernrate {LERNRATE} | Batch {BATCH_SIZE} | pos_weight: {NUTZE_GEWICHTUNG} | "
    f"Early Stopping: Val-Loss, Geduld {GEDULD}\n\n"
    "Kennzahlen (Mittel +/- Standardabweichung ueber Wiederholungen):\n" + "\n".join(zeilen) + "\n\n"
    "Pro Ziel:\n" + pro_ziel.to_string(index=False, float_format=lambda v: f"{v:.3f}") + "\n"
)
(RUN_DIR / "REPORT.txt").write_text(report, encoding="utf-8")
print(report)
print(f"Gespeichert unter: {RUN_DIR}")

Device: cpu | X (654, 1024), y (654, 14) | cross_validation, Seeds [42, 43, 44, 45, 46]
Run-Ordner: /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/Predictions/PyTorch_BCE_MLP_balanced/pdf_raw/max/mxbai-embed-large/recursive_1200c_200o_norefs/Run_1
  Wdh 1 Fold 1: beste Epoche 1 von 12
  Wdh 1 Fold 2: beste Epoche 2 von 13
  Wdh 1 Fold 3: beste Epoche 2 von 13
  Wdh 1 Fold 4: beste Epoche 2 von 13
  Wdh 1 Fold 5: beste Epoche 2 von 13
Wiederholung 1 (Seed 42): Makro-F1 0.570

  Wdh 2 Fold 1: beste Epoche 3 von 14
  Wdh 2 Fold 2: beste Epoche 3 von 14
  Wdh 2 Fold 3: beste Epoche 6 von 17
  Wdh 2 Fold 4: beste Epoche 3 von 14
  Wdh 2 Fold 5: beste Epoche 1 von 12
Wiederholung 2 (Seed 43): Makro-F1 0.576

  Wdh 3 Fold 1: beste Epoche 2 von 13
  Wdh 3 Fold 2: beste Epoche 1 von 12
  Wdh 3 Fold 3: beste Epoche 2 von 13
  Wdh 3 Fold 4: beste Epoche 1 von 12
  Wdh 3 Fold 5: beste Epoche 3 von 14
Wiederholung 3 (Seed 44): Makro-F1 0.568

  Wdh 4 Fold 1: beste Epoche 4 von 15
  Wdh 4 Fold 2:

In [ ]:
"""
Segment 7: Multi-Label-Klassifikation mit einem lokalen LLM (Ollama), Few-Shot.

Ablauf je Studie:
1. Text wie in Pipeline A aufbereiten (extract_pdf_structure: Literaturverzeichnis
   entfernt, gleiche Chunks).
2. Jeden Chunk einzeln vom LLM klassifizieren lassen (Liste zutreffender Labels).
3. Chunk-Stimmen zu einer Vorhersage pro Studie aggregieren:
   - Paper-Typen: Anteil der Chunks >= TYP_SCHWELLE (Typ praegt groessere Teile des Texts)
   - Artefakt-Ziele: mindestens ARTEFAKT_MIN_STIMMEN Chunks (Belege wie ein Code-Link
     stehen meist in ein bis zwei Chunks)
4. Optionale Regel aus dem Annotationsprotokoll: reine User Studies erhalten bei allen
   Artefakt-Zielen "No".

Die Rohantworten je Chunk werden gespeichert. Aggregation und Auswertung werden bei
jedem Lauf daraus neu berechnet; das LLM laeuft nur fuer Studien ohne passende
gespeicherte Antworten (gleiche Quelle, gleiche Lage des entfernten Literaturverzeichnisses,
gleicher Prompt, gleiches Modell, gleiche Optionen).

Ausgewertet werden alle Studien aus master_label_df mit denselben Metriken,
derselben Mehrheitsklassen-Basislinie und derselben Sensitivitaetsanalyse wie in
den MLP-Segmenten. Das LLM wird nicht trainiert; jede Studie wird genau einmal
vorhergesagt.
"""

import time

from sklearn.metrics import accuracy_score, hamming_loss, jaccard_score, precision_recall_fscore_support

# =============================================================================
# KONFIGURATION
# =============================================================================
LLM_MODELL = LLM_MODELL_NAME
PROMPT_TECHNIQUE = "few_shot_standard"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"

# Determinismus: Temperatur 0 und fester Seed; Kontextfenster explizit, damit Ollama
# lange Prompts nicht stillschweigend von vorne abschneidet
OLLAMA_OPTIONS = {"temperature": 0.0, "seed": GLOBAL_SEED, "num_ctx": 4096}
STRUKTURIERTE_AUSGABE = True    # Ollama erzwingt ein JSON-Array aus erlaubten Labels
LLM_TIMEOUT = 300               # Sekunden pro Chunk
LLM_MAX_VERSUCHE = 3

TYP_SCHWELLE = 0.10             # Anteil der Chunks fuer Paper-Typen
ARTEFAKT_MIN_STIMMEN = 1        # Chunks fuer die binaeren Ziele
USER_STUDY_REGEL = True

ZUSATZ_TARGETS = [t for t in PREDICTION_TARGETS if t != "paper_type"]
CATEGORIES = PAPER_TYPE_CATEGORIES + ZUSATZ_TARGETS   # gleiche Reihenfolge wie Segment 4

QUELL_ROOT, DATEI_ENDUNG = (MARKDOWN_ROOT, ".md") if DATEN_QUELLE == "markdown" else (STUDIEN_ROOT, ".pdf")
# Chunking-Bezeichnung ohne Embedding-Praefix: ein Wechsel des Embedding-Modells
# darf die (zeitaufwendigen) LLM-Ergebnisse nicht in einen neuen Ordner verschieben
LLM_STRATEGIE = f"recursive_{CHUNK_SIZE}c_{CHUNK_OVERLAP}o" + ("_norefs" if REFERENCES_ENTFERNEN else "")
TARGET_OUTPUT_ROOT = CURRENT_PREDICTIONS_PATH / PROMPT_TECHNIQUE / LLM_STRATEGIE

# =============================================================================
# PROMPT
# =============================================================================
# Definitionen nach Codebook (Criteria_Definitions) und Binarisierung im Cockpit
DEFINITIONEN = {
    "Algorithm Development": "The main focus is a novel algorithm for a known problem, e.g. optimization "
                             "techniques, machine learning models or a (deep) learning pipeline.",
    "Hardware Development": "The main contribution is novel or alternative hardware that is constructed or "
                            "designed and tested.",
    "Software Development": "The main contribution is a novel framework, app or other software, often shared "
                            "via a GitHub URL.",
    "User Study": "The paper primarily investigates human behavior, opinions or interactions through "
                  "experiments, surveys, interviews or observational studies.",
    "Dataset Paper": "The paper introduces a newly collected dataset that is explicitly released for use by "
                     "other researchers.",
    "Literature Review": "The paper systematically analyzes, surveys or synthesizes previously published "
                         "research without introducing new experimental results.",
    "code_required": "Executing code is necessary to reproduce the main results (not for purely theoretical "
                     "or descriptive results).",
    "code_available": "The required code is publicly available, completely or in part. Code available only "
                      "upon request does not count.",
    "code_url": "A direct URL to the published code repository (e.g. GitHub: https://github.com/) is given.",
    "data_required": "Data is required to reproduce the results, whether collected by the authors, "
                 "publicly available or proprietary.",
    "data_available": "The required data is publicly accessible, either via a given URL or because public "
                  "datasets are used, completely or in part. Data available only upon request does not count.",
    "data_url": "A direct URL to the dataset (e.g. GitHub: https://github.com/, Kaggle: https://www.kaggle.com/, OSF: https://osf.io/, Zenodo: https://zenodo.org/ ) is given.",
    "hardware_required": "Specific hardware beyond standard computing is required to reproduce the results.",
    "software_required": "Proprietary, paid or otherwise non-open-source software is required.",
}

BEISPIELE = [
    ("We designed a custom wearable sensor node using a flexible PCB layout. To evaluate the comfort, we "
     "recruited 15 participants for a user study. The dataset and firmware are available at "
     "https://github.com/project/core",
     ["Hardware Development", "User Study", "code_required", "code_available", "code_url",
      "data_required", "data_available", "data_url", "hardware_required"]),
    ("To minimize error, we introduce a novel loss function. The complete evaluation pipeline has been "
     "uploaded to our public GitHub repository as an open-source library.",
     ["Algorithm Development", "Software Development", "code_required", "code_available", "code_url"]),
    ("We present PillPal, a smartphone app for Android and iOS that reminds patients to take their "
     "medication and lets caregivers track adherence. The app consists of a React Native front end, a "
     "cloud-based scheduling service and a web dashboard for caregivers.",
     ["Software Development"]),
    ("Prior work has examined sleep tracking in clinical settings [12, 15], while others focused on "
     "consumer devices [3].",
     []),
]


def baue_prompt_vorlage() -> str:
    labels = "\n".join(f"- {k}: {DEFINITIONEN[k]}" for k in CATEGORIES)
    beispiele = "\n\n".join(
        f"Example {i}:\nText: \"{text}\"\nResponse: {json.dumps(antwort)}"
        for i, (text, antwort) in enumerate(BEISPIELE, start=1)
    )
    return (
        "You are an expert classifier for the reproducibility of research papers.\n"
        "You receive one text chunk of a paper. Return the labels for which this chunk itself "
        "provides evidence. Several labels may apply. If the chunk provides no evidence for any "
        "label, return an empty array.\n\n"
        f"Allowed labels and their definitions:\n{labels}\n\n"
        f"{beispiele}\n\n"
        "Respond ONLY with a JSON array of strings from the allowed labels. "
        "No explanation, no markdown.\n\n"
        "Text chunk:\n{chunk}"
    )


PROMPT_VORLAGE = baue_prompt_vorlage()
ANTWORT_SCHEMA = {"type": "array", "items": {"type": "string", "enum": CATEGORIES}}
_KANONISCH = {c.lower(): c for c in CATEGORIES}


# =============================================================================
# LLM-AUFRUF
# =============================================================================
def parse_labels(text: str):
    """JSON-Array aus der Antwort lesen; Gross-/Kleinschreibung tolerieren, unbekannte Labels verwerfen."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    treffer = re.search(r"\[.*\]", text, re.DOTALL)
    if not treffer:
        return None
    try:
        liste = json.loads(treffer.group(0))
    except json.JSONDecodeError:
        return None
    if not isinstance(liste, list):
        return None
    return sorted({_KANONISCH[s.strip().lower()] for s in liste
                   if isinstance(s, str) and s.strip().lower() in _KANONISCH})


def klassifiziere_chunk(text: str):
    """Rueckgabe: (Labels oder None bei Fehler, Anzahl Prompt-Tokens)."""
    anfrage = {"model": LLM_MODELL, "prompt": PROMPT_VORLAGE.replace("{chunk}", text),
               "stream": False, "options": OLLAMA_OPTIONS}
    if STRUKTURIERTE_AUSGABE:
        anfrage["format"] = ANTWORT_SCHEMA
    for versuch in range(LLM_MAX_VERSUCHE):
        try:
            antwort = requests.post(OLLAMA_GENERATE_URL, json=anfrage, timeout=LLM_TIMEOUT)
            if antwort.status_code == 200:
                daten = antwort.json()
                labels = parse_labels(daten.get("response", ""))
                if labels is not None:
                    return labels, daten.get("prompt_eval_count")
            else:
                print(f"LLM-Fehler (HTTP {antwort.status_code}): {antwort.text[:200]}")
        except (requests.RequestException, ValueError) as e:
            print(f"LLM-Fehler: {e}")
        time.sleep(2 ** versuch)
    return None, None


def llm_digest() -> str:
    kandidaten = {LLM_MODELL, f"{LLM_MODELL}:latest"}
    for name, digest in get_ollama_model_info().items():
        if name in kandidaten:
            return digest
    return "unbekannt"


# =============================================================================
# AGGREGATION UND LABELS
# =============================================================================
def aggregiere(chunk_labels: list) -> list:
    n = len(chunk_labels)
    stimmen = {c: sum(c in labels for labels in chunk_labels) for c in CATEGORIES}
    pred = [int(stimmen[c] / n >= TYP_SCHWELLE) if c in PAPER_TYPE_CATEGORIES
            else int(stimmen[c] >= ARTEFAKT_MIN_STIMMEN) for c in CATEGORIES]
    if USER_STUDY_REGEL:
        typen = {c for c, p in zip(CATEGORIES, pred) if p and c in PAPER_TYPE_CATEGORIES}
        if typen == {"User Study"}:
            pred = [p if c in PAPER_TYPE_CATEGORIES else 0 for c, p in zip(CATEGORIES, pred)]
    return pred


def kodiere_labels(row) -> list:
    """Wie Segment 4: Multi-Hot fuer paper_type, 0/1 fuer die binaeren Ziele."""
    typen = {t.strip() for t in str(row["paper_type"]).split(";") if t.strip()}
    return [int(k in typen) for k in PAPER_TYPE_CATEGORIES] + [int(row[z] == "Yes") for z in ZUSATZ_TARGETS]


# =============================================================================
# HAUPTSCHLEIFE
# =============================================================================
if not check_ollama():
    raise RuntimeError("Ollama-Server nicht erreichbar.")
DIGEST = llm_digest()
if DIGEST == "unbekannt":
    raise RuntimeError(f"Modell '{LLM_MODELL}' ist in Ollama nicht installiert: ollama pull {LLM_MODELL}")

# Alles, was die Rohantworten beeinflusst; Aggregationsparameter gehoeren nicht dazu
LLM_CONFIG = {
    "modell": LLM_MODELL, "digest": DIGEST, "optionen": OLLAMA_OPTIONS,
    "strukturiert": STRUKTURIERTE_AUSGABE, "prompt_sha256": hashlib.sha256(PROMPT_VORLAGE.encode()).hexdigest(),
    "daten_quelle": DATEN_QUELLE, "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP,
    "separators": CHUNK_SEPARATORS, "references_entfernen": REFERENCES_ENTFERNEN,
}
LLM_CONFIG_SHA256 = hashlib.sha256(json.dumps(LLM_CONFIG, sort_keys=True).encode()).hexdigest()

studien = master_label_df.sort_values("paper_id").to_dict("records")
print(f"LLM: {LLM_MODELL} ({DIGEST[:12]}) | {len(studien)} Studien | Ziel: {TARGET_OUTPUT_ROOT}")
print(f"Prompt: {len(PROMPT_VORLAGE)} Zeichen ohne Chunk | Optionen: {OLLAMA_OPTIONS}")

y_true, y_pred, scores, paper_ids, fehler, gekuerzt = [], [], [], [], [], []

for row in tqdm(studien, desc="LLM"):
    quelle = QUELL_ROOT / row["year"] / f"{row['paper_id']}{DATEI_ENDUNG}"
    out_path = TARGET_OUTPUT_ROOT / row["year"] / f"{row['paper_id']}_prediction.json"
    if not quelle.exists():
        fehler.append((row["paper_id"], "Quelldatei fehlt"))
        continue
    quelle_sha = file_sha256(quelle)
    chunks, meta = extract_pdf_structure(quelle)

    gespeichert = None
    if out_path.exists():
        alt = json.loads(out_path.read_text(encoding="utf-8"))
        if (alt.get("llm_config_sha256") == LLM_CONFIG_SHA256 and alt.get("source_sha256") == quelle_sha
                and schnitt_signatur(alt.get("extraction_meta", {})) == schnitt_signatur(meta)):
            gespeichert = alt

    if gespeichert is None:
        chunk_labels, max_tokens, ok = [], 0, True
        for chunk in chunks:
            labels, tokens = klassifiziere_chunk(chunk["text"])
            if labels is None:
                ok = False
                break
            chunk_labels.append(labels)
            max_tokens = max(max_tokens, tokens or 0)
        # Alle Chunks oder keiner, wie bei den Embeddings
        if not ok or not chunks:
            fehler.append((row["paper_id"], "LLM-Antwort fehlt" if chunks else "keine Chunks"))
            continue
        gespeichert = {
            "paper_id": row["paper_id"],
            "llm_config": LLM_CONFIG, "llm_config_sha256": LLM_CONFIG_SHA256,
            "source_sha256": quelle_sha, "extraction_meta": meta,
            "max_prompt_tokens": max_tokens,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "chunk_labels": chunk_labels,
        }
        out_path.parent.mkdir(parents=True, exist_ok=True)
        tmp = out_path.with_suffix(".json.tmp")
        tmp.write_text(json.dumps(gespeichert, ensure_ascii=False), encoding="utf-8")
        tmp.replace(out_path)

    if (gespeichert.get("max_prompt_tokens") or 0) >= OLLAMA_OPTIONS["num_ctx"]:
        gekuerzt.append(row["paper_id"])

    chunk_labels = gespeichert["chunk_labels"]
    y_true.append(kodiere_labels(row))
    y_pred.append(aggregiere(chunk_labels))
    scores.append([sum(c in l for l in chunk_labels) / len(chunk_labels) for c in CATEGORIES])
    paper_ids.append(row["paper_id"])

y_true, y_pred, scores = np.array(y_true), np.array(y_pred), np.array(scores)
if fehler:
    print(f"Nicht ausgewertet: {len(fehler)} Studien, z. B. {fehler[:5]}")
if gekuerzt:
    print(f"WARNUNG: Prompt erreichte num_ctx bei {len(gekuerzt)} Studien (moeglicherweise gekuerzt): {gekuerzt[:10]}")

# =============================================================================
# AUSWERTUNG (gleiche Kennzahlen wie die MLP-Segmente)
# =============================================================================
def metriken(y_wahr, y_vorh) -> dict:
    p, r, f, s = precision_recall_fscore_support(y_wahr, y_vorh, average=None, zero_division=0)
    return {"precision": p, "recall": r, "f1": f, "support": s, "macro_f1": float(f.mean()),
            "hamming_loss": hamming_loss(y_wahr, y_vorh), "subset_accuracy": accuracy_score(y_wahr, y_vorh),
            "jaccard_samples": jaccard_score(y_wahr, y_vorh, average="samples", zero_division=0)}


typ_roh = master_label_raw_df.set_index("paper_id")["paper_type"].astype(str).str.strip()
reine_us = np.array([typ_roh.get(p) == "User Study" for p in paper_ids])
# Ohne Training: Mehrheitsklasse je Ziel ueber alle ausgewerteten Studien
baseline = np.tile((y_true.mean(axis=0) >= 0.5).astype(int), (len(y_true), 1))

ergebnis = {
    "modell": metriken(y_true, y_pred),
    "baseline": metriken(y_true, baseline),
    "modell_ohne_reine_us": metriken(y_true[~reine_us], y_pred[~reine_us]),
}

RUN_BASE = TARGET_OUTPUT_ROOT / "auswertungen"
RUN_BASE.mkdir(parents=True, exist_ok=True)
nummern = [int(d.name.split("_")[1]) for d in RUN_BASE.glob("Run_*") if d.name.split("_")[1].isdigit()]
RUN_DIR = RUN_BASE / f"Run_{max(nummern, default=0) + 1}"
RUN_DIR.mkdir()

pro_ziel = pd.DataFrame({
    "ziel": CATEGORIES, "support": ergebnis["modell"]["support"],
    "precision": ergebnis["modell"]["precision"], "recall": ergebnis["modell"]["recall"],
    "f1": ergebnis["modell"]["f1"], "f1_mehrheitsklasse": ergebnis["baseline"]["f1"],
    "f1_ohne_reine_user_studies": ergebnis["modell_ohne_reine_us"]["f1"],
})
pro_ziel.to_csv(RUN_DIR / "metriken_pro_ziel.csv", index=False)

zeilen = []
for i, p_id in enumerate(paper_ids):
    zeile = {"paper_id": p_id, "reine_user_study": bool(reine_us[i])}
    for j, c in enumerate(CATEGORIES):
        zeile[f"{c}_true"] = int(y_true[i, j])
        zeile[f"{c}_anteil_chunks"] = round(float(scores[i, j]), 4)
        zeile[f"{c}_pred"] = int(y_pred[i, j])
    zeilen.append(zeile)
pd.DataFrame(zeilen).to_csv(RUN_DIR / "vorhersagen.csv", index=False)

gesamt = {f"{k}.{w}": float(v[w]) for k, v in ergebnis.items()
          for w in ["macro_f1", "hamming_loss", "subset_accuracy", "jaccard_samples"]}
konfiguration = {
    "klassifikator": f"LLM_{LLM_MODELL}_{PROMPT_TECHNIQUE}",
    "llm_config": LLM_CONFIG, "llm_config_sha256": LLM_CONFIG_SHA256,
    "aggregation": {"typ_schwelle": TYP_SCHWELLE, "artefakt_min_stimmen": ARTEFAKT_MIN_STIMMEN,
                    "user_study_regel": USER_STUDY_REGEL},
    "prompt_vorlage": PROMPT_VORLAGE,
    "n_ausgewertet": len(paper_ids), "nicht_ausgewertet": fehler, "moeglich_gekuerzt": gekuerzt,
    "klassifikation_csv_sha256": KLASSIFIKATION_CSV_SHA256,
    "pred_sha256": hashlib.sha256(y_pred.tobytes()).hexdigest(),
    "ergebnis": gesamt,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(konfiguration, indent=2, default=str), encoding="utf-8")
save_run_config(extra={"llm_run": str(RUN_DIR), **{k: v for k, v in konfiguration.items() if k != "prompt_vorlage"}})

report = (
    f"LLM: {LLM_MODELL} ({DIGEST[:12]}) | {PROMPT_TECHNIQUE} | {DATEN_QUELLE} | {LLM_STRATEGIE}\n"
    f"Aggregation: Typen >= {TYP_SCHWELLE:.0%} der Chunks, Artefakte >= {ARTEFAKT_MIN_STIMMEN} Chunk(s), "
    f"User-Study-Regel: {USER_STUDY_REGEL}\n"
    f"Ausgewertet: {len(paper_ids)} Studien\n\n"
    + "\n".join(f"{k:<38} {v:.4f}" for k, v in gesamt.items()) + "\n\n"
    + pro_ziel.to_string(index=False, float_format=lambda v: f"{v:.3f}") + "\n" 
)
(RUN_DIR / "REPORT.txt").write_text(report, encoding="utf-8")
print(report)
print(f"Gespeichert unter: {RUN_DIR}")

LLM: llama3.2 (a80c4f17acd5) | 654 Studien | Ziel: /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/predictions_llm/pdf_raw/llama3.2/few_shot_standard/recursive_1200c_200o_norefs
Prompt: 3474 Zeichen ohne Chunk | Optionen: {'temperature': 0.0, 'seed': 42, 'num_ctx': 4096}


LLM:  47%|████████████████▉                   | 307/654 [00:59<01:36,  3.60it/s]

In [32]:
"""
RandomForest_MultiOutput

Zweck dieser Zelle:
Klassifikation der Paper-Typen und Metadaten (16 Labels).
LÖSUNG FÜR KLASSEN-UNGLEICHGEWICHT: Nutzt den MultiOutputClassifier in Kombination 
mit Random Forests (Binary Relevance). So wird für jedes der 16 Labels ein eigener 
Spezialist trainiert, der das Klassen-Ungleichgewicht isoliert ausgleichen kann.

"""

import numpy as np
import json
import pandas as pd  
import random
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, hamming_loss, accuracy_score, jaccard_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# --- CONFIG FÜR DIE EVALUIERUNG ---
VALIDIERUNGS_METHODE = "cross_validation" # "holdout" oder "cross_validation"
N_FOLDS = 5                              

# DETERMINISTISCHER START
def set_deterministic_seeds(seed=42):
    """Fixiert alle Zufallsgeneratoren für absolute Reproduzierbarkeit."""
    random.seed(seed)
    np.random.seed(seed)
    print(f"🔒 Zufalls-Seed auf '{seed}' fixiert. Ergebnisse sind deterministisch!")

seed_to_use = globals().get('GLOBAL_SEED', 42)
set_deterministic_seeds(seed_to_use)

# --- 1. DATEN LADEN ---
X = np.load(CURRENT_FUSION_PATH / 'X_data.npy')
y = np.load(CURRENT_FUSION_PATH / 'y_data.npy')

with open(CURRENT_FUSION_PATH / 'y_columns.json', 'r', encoding='utf-8') as f:
    ML_COLUMNS = json.load(f)

# --- 2. AUTOMATISCHE RUN-ORDNER LOGIK ---
# 🔄 GEÄNDERT: Statischer Name, damit alle Runs (Holdout & CrossVal) im selben Ordner landen!
classifier_name = "RandomForest_MultiOutput" 

base_dir = PREDICTIONS_ML_ROOT / classifier_name / DATEN_QUELLE / POOLING_STRATEGIE / MODELL_NAME / STRATEGIE_NAME
base_dir.mkdir(parents=True, exist_ok=True)

existing_runs = [d.name for d in base_dir.iterdir() if d.is_dir() and d.name.startswith("Run_")]
run_numbers = [int(name.split("_")[1]) for name in existing_runs if name.split("_")[1].isdigit()]
next_run = max(run_numbers) + 1 if run_numbers else 1

RUN_DIR = base_dir / f"Run_{next_run}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"📊 Daten geladen (Studien: {X.shape[0]}, Features: {X.shape[1]}, Labels: {y.shape[1]})")
print(f"⚙️ Gewählte Evaluierung: {VALIDIERUNGS_METHODE.upper()}")
print(f"📁 Neuer Arbeitsordner erstellt: {RUN_DIR}\n")

# --- HEADER FÜR DIE TXT-DATEI VORBEREITEN ---
report_header = (
    f"=== RUN PARAMETER ===\n"
    f"Datenquelle: {DATEN_QUELLE}\n"
    f"Embed-Modell: {MODELL_NAME}\n"
    f"Chunking-Strategie: {STRATEGIE_NAME}\n"
    f"Pooling: {POOLING_STRATEGIE}\n"
    f"Evaluierung: {VALIDIERUNGS_METHODE}\n"
    f"Klassifikator: {classifier_name}\n"
    f"Zufalls-Seed: {seed_to_use}\n"
    f"Feature Scaling: StandardScaler (für Pipeline-Konsistenz)\n"
    f"Architektur: Binary Relevance (16 separierte RF-Modelle)\n"
    f"Bäume pro Label: 150 (max_depth=15)\n"
    f"Class Weighting: Balanced\n"
    f"=====================\n\n"
)

# --- 3. MODELL DEFINIEREN (Das Spezialisten-Team) ---
def create_specialist_team():
    base_rf = RandomForestClassifier(
        n_estimators=150,           # 150 Entscheidungsbäume pro Label
        class_weight="balanced",    # Gleicht seltene Klassen automatisch aus
        max_depth=15,               # Verhindert zu starkes Overfitting
        random_state=seed_to_use,
        n_jobs=-1                   # Nutzt alle CPU-Kerne
    )
    return MultiOutputClassifier(base_rf)

# --- 4. EVALUIERUNG - LOGIK ---

if VALIDIERUNGS_METHODE == "holdout":
    # --- A: KLASSISCHER HOLDOUT SPLIT (80/20) ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed_to_use)
    
    # Skalierung (Für RF nicht strikt nötig, aber sichert 100% Vergleichbarkeit der Pipelines)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    clf = create_specialist_team()
    print("🏋️ Starte einmaliges Training des RF-Spezialisten-Teams (Holdout)...")
    clf.fit(X_train, y_train)
    
    predictions = clf.predict(X_test)
    
    # Auswertung
    report_str = classification_report(y_test, predictions, target_names=ML_COLUMNS, zero_division=0)
    h_loss = hamming_loss(y_test, predictions)
    exact_match = accuracy_score(y_test, predictions)
    jaccard = jaccard_score(y_test, predictions, average='samples', zero_division=0)
    
    log_text = f"""📊 KLASSIFIZIERUNGS-BERICHT (Holdout 80/20):
=================================================================
{report_str}
=================================================================
📉 Hamming Loss: {h_loss:.4f}
🎯 Subset Accuracy (Strenge Metrik): {exact_match:.4f} ({exact_match*100:.1f}%)
🤝 Jaccard Similarity (Teilpunkte):  {jaccard:.4f} ({jaccard*100:.1f}%)
"""
    print(log_text)
    
    # --- DATEN LOKAL IM RUN-ORDNER SPEICHERN ---
    daten_dict = {}
    for i, col_name in enumerate(ML_COLUMNS):
        daten_dict[f"{col_name}_true"] = y_test[:, i]
        daten_dict[f"{col_name}_pred"] = predictions[:, i]
    pd.DataFrame(daten_dict).to_csv(RUN_DIR / f"Run_{next_run}.csv", index=False)
    
    with open(RUN_DIR / f"Run_{next_run}.txt", "w", encoding="utf-8") as f:
        f.write(report_header + log_text)
        
    print(f"\n💾 ALLE DATEN ERFOLGREICH GESPEICHERT!")
    print(f"📂 Zu finden unter: {RUN_DIR}")

elif VALIDIERUNGS_METHODE == "cross_validation":
    # --- B: N-FOLD CROSS VALIDATION ---
    print(f"🔄 Starte {N_FOLDS}-Fold Multi-Label Cross-Validation mit isolierten Random Forests...")
    
    mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed_to_use)
    
    oof_predictions = np.zeros_like(y)
    hamming_losses = []
    
    for fold, (train_idx, test_idx) in enumerate(mskf.split(X, y)):
        print(f"  🌲 Wachse Wald für Fold {fold + 1}/{N_FOLDS}...")
        
        X_train_fold, X_test_fold = X[train_idx], X[test_idx]
        y_train_fold, y_test_fold = y[train_idx], y[test_idx]
        
        # Skalierung pro Fold (verhindert Data Leakage!)
        scaler = StandardScaler()
        X_train_fold = scaler.fit_transform(X_train_fold)
        X_test_fold = scaler.transform(X_test_fold)
        
        clf_fold = create_specialist_team()
        clf_fold.fit(X_train_fold, y_train_fold)
        
        fold_preds = clf_fold.predict(X_test_fold)
        oof_predictions[test_idx] = fold_preds
        hamming_losses.append(hamming_loss(y_test_fold, fold_preds))
    
    # Auswertung zusammenbauen
    report_str = classification_report(y, oof_predictions, target_names=ML_COLUMNS, zero_division=0)
    mean_hamming = np.mean(hamming_losses)
    std_hamming = np.std(hamming_losses)
    exact_match = accuracy_score(y, oof_predictions)
    jaccard = jaccard_score(y, oof_predictions, average='samples', zero_division=0)
    
    log_text = f"""📊 REPRODUZIERBARER GESAMT-BERICHT OVER ALL FOLDS (Binary Relevance Random Forest):
=================================================================
{report_str}
=================================================================
📉 Mittlerer Hamming Loss: {mean_hamming:.4f} (± {std_hamming:.4f})
🎯 Subset Accuracy (Strenge Metrik): {exact_match:.4f} ({exact_match*100:.1f}%)
🤝 Jaccard Similarity (Teilpunkte):  {jaccard:.4f} (Das Modell trifft im Schnitt {jaccard*100:.1f}% der relevanten Labels pro Studie!)
"""
    print("\n" + log_text)

    # --- DATEN LOKAL IM RUN-ORDNER SPEICHERN ---
    daten_dict = {}
    for i, col_name in enumerate(ML_COLUMNS):
        daten_dict[f"{col_name}_true"] = y[:, i]
        daten_dict[f"{col_name}_pred"] = oof_predictions[:, i]
    pd.DataFrame(daten_dict).to_csv(RUN_DIR / f"Run_{next_run}.csv", index=False)
    
    with open(RUN_DIR / f"Run_{next_run}.txt", "w", encoding="utf-8") as f:
        f.write(report_header + log_text)
        
    print(f"💾 ALLE DATEN ERFOLGREICH GESPEICHERT!")
    print(f"📂 Zu finden unter: {RUN_DIR}")

🔒 Zufalls-Seed auf '42' fixiert. Ergebnisse sind deterministisch!
📊 Daten geladen (Studien: 654, Features: 768, Labels: 16)
⚙️ Gewählte Evaluierung: CROSS_VALIDATION
📁 Neuer Arbeitsordner erstellt: /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/Predictions/RandomForest_MultiOutput/markdown/max/nomic-embed-text-v2-moe/1200_chars_fixed_chunks/Run_8

🔄 Starte 5-Fold Multi-Label Cross-Validation mit isolierten Random Forests...
  🌲 Wachse Wald für Fold 1/5...
  🌲 Wachse Wald für Fold 2/5...
  🌲 Wachse Wald für Fold 3/5...
  🌲 Wachse Wald für Fold 4/5...
  🌲 Wachse Wald für Fold 5/5...

📊 REPRODUZIERBARER GESAMT-BERICHT OVER ALL FOLDS (Binary Relevance Random Forest):
                       precision    recall  f1-score   support

Algorithm Development       0.78      1.00      0.87       498
 Hardware Development       0.75      0.02      0.03       174
 Software Development       0.56      0.77      0.65       351
           User Study       0.71      0.90      0.79       389
        D

In [10]:
"""
RandomForest_Balanced_Thresh

Zweck dieser Zelle:
Multi-Label-Klassifikation der 16 Paper-Typen und Metadaten mittels Random Forest.
LÖSUNG FÜR KLASSEN-UNGLEICHGEWICHT: Nutzt `class_weight='balanced'`.

🔄 STANDARDISIERT: Inklusive deterministischem Seed, N-Fold Cross-Validation, 
robuster predict_proba Auswertung und automatischer Run-Ordner-Generierung.
(Hinweis: Random Forests nutzen keine Epochen, daher entfallen Lernkurven und Checkpointing).
"""

import numpy as np
import json
import pandas as pd  
import random
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, hamming_loss, accuracy_score, jaccard_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# --- CONFIG FÜR DIE EVALUIERUNG & ARCHITEKTUR ---
VALIDIERUNGS_METHODE = "cross_validation" # "holdout" oder "cross_validation"
N_FOLDS = 5                              
THRESHOLD = 0.5                           # 🔄 KORRIGIERT: Zurück auf 50%. Die Kombi aus 30% und 'balanced' erzeugte zu viele False Positives!
N_ESTIMATORS = 300                        # Anzahl der Bäume
MAX_DEPTH = 20                            # Verhindert tiefes Overfitting (Alternative zur L2-Regulierung)

# DETERMINISTISCHER START
def set_deterministic_seeds(seed=42):
    """Fixiert alle Zufallsgeneratoren für absolute Reproduzierbarkeit."""
    random.seed(seed)
    np.random.seed(seed)
    print(f"🔒 Zufalls-Seed auf '{seed}' fixiert. Ergebnisse sind deterministisch!")

seed_to_use = globals().get('GLOBAL_SEED', 42)
set_deterministic_seeds(seed_to_use)

# --- 1. DATEN VOM AKTUELLEN STRATEGIE-ORDNER LADEN ---
X = np.load(CURRENT_FUSION_PATH / 'X_data.npy')
y = np.load(CURRENT_FUSION_PATH / 'y_data.npy')

with open(CURRENT_FUSION_PATH / 'y_columns.json', 'r', encoding='utf-8') as f:
    ML_COLUMNS = json.load(f)

# --- 2. AUTOMATISCHE RUN-ORDNER LOGIK ---
classifier_id = "RandomForest_Balanced_Thresh"

# Die tiefe Verzeichnisstruktur analog zu den Fusionsdaten
base_dir = PREDICTIONS_ML_ROOT / classifier_id / DATEN_QUELLE / POOLING_STRATEGIE / MODELL_NAME / STRATEGIE_NAME
base_dir.mkdir(parents=True, exist_ok=True)

# Scanne den Ordner nach existierenden "Run_X" Ordnern und finde die höchste Nummer
existing_runs = [d.name for d in base_dir.iterdir() if d.is_dir() and d.name.startswith("Run_")]
run_numbers = [int(name.split("_")[1]) for name in existing_runs if name.split("_")[1].isdigit()]
next_run = max(run_numbers) + 1 if run_numbers else 1

# Erstelle den neuen Arbeitsordner für diesen Durchlauf
RUN_DIR = base_dir / f"Run_{next_run}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"📊 Daten erfolgreich geladen (Studien: {X.shape[0]}, Pooling-Dimension: {X.shape[1]})")
print(f"⚙️ Gewählte Evaluierung: {VALIDIERUNGS_METHODE.upper()}")
print(f"🌲 Architektur: Random Forest (Bäume: {N_ESTIMATORS}, Max-Tiefe: {MAX_DEPTH})")
print(f"⚖️ Gewichtung: class_weight='balanced'")
print(f"🎚️ Thresholding: {THRESHOLD*100}% für positive Klassifikation")
print(f"📁 Neuer Arbeitsordner erstellt: {RUN_DIR}\n")

# --- HEADER FÜR DIE TXT-DATEI VORBEREITEN ---
report_header = (
    f"=== RUN PARAMETER ===\n"
    f"Datenquelle: {DATEN_QUELLE}\n"
    f"Embed-Modell: {MODELL_NAME}\n"
    f"Chunking-Strategie: {STRATEGIE_NAME}\n"
    f"Pooling: {POOLING_STRATEGIE}\n"
    f"Evaluierung: {VALIDIERUNGS_METHODE}\n"
    f"Klassifikator: {classifier_id}\n"
    f"Zufalls-Seed: {seed_to_use}\n"
    f"Bäume (n_estimators): {N_ESTIMATORS}\n"
    f"Max Depth: {MAX_DEPTH}\n"
    f"Class Weighting: Balanced\n"
    f"Decision Threshold: {THRESHOLD}\n"
    f"=====================\n\n"
)

# --- 3. MODELL-PROTOTYP & ROBUSTE THRESHOLD FUNKTION ---
def create_new_rf():
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS, 
        max_depth=MAX_DEPTH,      
        class_weight='balanced', 
        random_state=seed_to_use,
        n_jobs=-1 # Nutzt alle CPU-Kerne           
    )

def apply_custom_threshold(rf_model, X_input, threshold):
    """
    Sichere Extraktion der Wahrscheinlichkeiten und Anwendung des Custom-Thresholds.
    Fängt Scikit-Learn Eigenheiten bei komplett fehlenden Klassen in einem CV-Fold ab.
    """
    probs_list = rf_model.predict_proba(X_input)
    predictions = np.zeros((X_input.shape[0], len(ML_COLUMNS)))
    
    for i in range(len(ML_COLUMNS)):
        # Prüfen, ob das Modell für dieses Label überhaupt beide Klassen (0 und 1) gesehen hat
        if len(rf_model.classes_[i]) == 2:
            prob_pos = probs_list[i][:, 1] # Wahrscheinlichkeit für "Yes" (Klasse 1)
        else:
            # Falls in einem extrem unausgeglichenen Fold nur 0en vorkamen
            class_val = rf_model.classes_[i][0]
            prob_pos = np.ones(X_input.shape[0]) if class_val == 1 else np.zeros(X_input.shape[0])
            
        # Anwenden des 50% Thresholds
        predictions[:, i] = (prob_pos >= threshold).astype(int)
        
    return predictions


# --- 4. EVALUIERUNG - LOGIK ---
if VALIDIERUNGS_METHODE == "holdout":
    # --- A: KLASSISCHER HOLDOUT SPLIT ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed_to_use)
    
    rf = create_new_rf()
    print("🏋️ Starte RandomForest Training (Holdout Standard)...")
    rf.fit(X_train, y_train)
    
    # Vorhersage mit Custom Thresholding (z.B. 0.5)
    predictions = apply_custom_threshold(rf, X_test, THRESHOLD)
    
    # Auswertung
    report_str = classification_report(y_test, predictions, target_names=ML_COLUMNS, zero_division=0)
    h_loss = hamming_loss(y_test, predictions)
    exact_match = accuracy_score(y_test, predictions)
    jaccard = jaccard_score(y_test, predictions, average='samples', zero_division=0)
    
    log_text = f"""📊 KLASSIFIZIERUNGS-BERICHT (Holdout 80/20):
=================================================================
{report_str}
=================================================================
📉 Hamming Loss: {h_loss:.4f}
🎯 Subset Accuracy (Strenge Metrik): {exact_match:.4f} ({exact_match*100:.1f}%)
🤝 Jaccard Similarity (Teilpunkte):  {jaccard:.4f} ({jaccard*100:.1f}%)
"""
    print(log_text)
    
    # --- DATEN LOKAL IM RUN-ORDNER SPEICHERN ---
    daten_dict = {}
    for i, col_name in enumerate(ML_COLUMNS):
        daten_dict[f"{col_name}_true"] = y_test[:, i]
        daten_dict[f"{col_name}_pred"] = predictions[:, i]
    pd.DataFrame(daten_dict).to_csv(RUN_DIR / f"Run_{next_run}.csv", index=False)
    
    with open(RUN_DIR / f"Run_{next_run}.txt", "w", encoding="utf-8") as f:
        f.write(report_header + log_text)
        
    print(f"\n💾 ALLE DATEN ERFOLGREICH GESPEICHERT!")
    print(f"📂 Zu finden unter: {RUN_DIR}")

elif VALIDIERUNGS_METHODE == "cross_validation":
    # --- B: N-FOLD CROSS VALIDATION ---
    print(f"🔄 Starte {N_FOLDS}-Fold Multi-Label Cross-Validation...")
    
    mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed_to_use)
    
    oof_predictions = np.zeros_like(y)
    hamming_losses = []
    
    for fold, (train_idx, test_idx) in enumerate(mskf.split(X, y)):
        print(f"  🌲 Wachse Wald für Fold {fold + 1}/{N_FOLDS}...")
        
        X_train_fold, X_test_fold = X[train_idx], X[test_idx]
        y_train_fold, y_test_fold = y[train_idx], y[test_idx]
        
        rf_fold = create_new_rf()
        rf_fold.fit(X_train_fold, y_train_fold)
        
        # Vorhersage mit Custom Thresholding (z.B. 0.5)
        fold_preds = apply_custom_threshold(rf_fold, X_test_fold, THRESHOLD)
        
        oof_predictions[test_idx] = fold_preds
        hamming_losses.append(hamming_loss(y_test_fold, fold_preds))
    
    # Auswertung
    report_str = classification_report(y, oof_predictions, target_names=ML_COLUMNS, zero_division=0)
    mean_hamming = np.mean(hamming_losses)
    std_hamming = np.std(hamming_losses)
    exact_match = accuracy_score(y, oof_predictions)
    jaccard = jaccard_score(y, oof_predictions, average='samples', zero_division=0)
    
    log_text = f"""📊 REPRODUZIERBARER GESAMT-BERICHT OVER ALL FOLDS (Cross-Validation):
=================================================================
{report_str}
=================================================================
📉 Mittlerer Hamming Loss: {mean_hamming:.4f} (± {std_hamming:.4f})
🎯 Subset Accuracy (Strenge Metrik): {exact_match:.4f} ({exact_match*100:.1f}%)
🤝 Jaccard Similarity (Teilpunkte):  {jaccard:.4f} (Das Modell trifft im Schnitt {jaccard*100:.1f}% der relevanten Labels pro Studie!)
"""
    print("\n" + log_text)
    
    # --- DATEN LOKAL IM RUN-ORDNER SPEICHERN ---
    daten_dict = {}
    for i, col_name in enumerate(ML_COLUMNS):
        daten_dict[f"{col_name}_true"] = y[:, i]
        daten_dict[f"{col_name}_pred"] = oof_predictions[:, i]
    pd.DataFrame(daten_dict).to_csv(RUN_DIR / f"Run_{next_run}.csv", index=False)
    
    with open(RUN_DIR / f"Run_{next_run}.txt", "w", encoding="utf-8") as f:
        f.write(report_header + log_text)
        
    print(f"💾 ALLE DATEN ERFOLGREICH GESPEICHERT!")
    print(f"📂 Zu finden unter: {RUN_DIR}")

🔒 Zufalls-Seed auf '42' fixiert. Ergebnisse sind deterministisch!
📊 Daten erfolgreich geladen (Studien: 654, Pooling-Dimension: 768)
⚙️ Gewählte Evaluierung: CROSS_VALIDATION
🌲 Architektur: Random Forest (Bäume: 300, Max-Tiefe: 20)
⚖️ Gewichtung: class_weight='balanced'
🎚️ Thresholding: 50.0% für positive Klassifikation
📁 Neuer Arbeitsordner erstellt: /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/Predictions/RandomForest_Balanced_Thresh/markdown/max/nomic-embed-text-v2-moe/1200_chars_fixed_chunks/Run_2

🔄 Starte 5-Fold Multi-Label Cross-Validation...
  🌲 Wachse Wald für Fold 1/5...
  🌲 Wachse Wald für Fold 2/5...
  🌲 Wachse Wald für Fold 3/5...
  🌲 Wachse Wald für Fold 4/5...
  🌲 Wachse Wald für Fold 5/5...

📊 REPRODUZIERBARER GESAMT-BERICHT OVER ALL FOLDS (Cross-Validation):
                       precision    recall  f1-score   support

Algorithm Development       0.76      1.00      0.86       498
 Hardware Development       0.00      0.00      0.00       174
 Software Developme

In [11]:
"""
Zweck dieser Zelle:
Robuste 16-Label Few-Shot-Klassifikation mit Llama 3.2 (Version 3.0) via Ollama.
Technik: Standard Few-Shot OHNE spezifische Keyword-Guidelines.
Speichert die Resultate sauber im vordefinierten CURRENT_PREDICTIONS_PATH ab.
Nutzt eine prozentuale Schwellenwert-Aggregation, um echtes Multi-Label-Verhalten zu erzwingen.
"""

import requests
import json
import os
import re
import datetime
import numpy as np
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report

# --- 1. CONFIG & PFADE AUS DEM COCKPIT ---
LLM_LIMIT = 600 
OLLAMA_GENERATE_URL = "http://localhost:11434/api/generate"
LLM_MODELL = LLM_MODELL_NAME  # Nutzt die globale Variable aus deinem Cockpit

# 🔄 GEÄNDERT: Neue Prompt-Technik ohne Keywords
PROMPT_TECHNIQUE = "few_shot_standard"

# Dynamische Bestimmung der Dateiendung
DATEI_ENDUNG = ".md" if DATEN_QUELLE == "markdown" else ".pdf"
QUELL_ORDNER_ROOT = MARKDOWN_ROOT if DATEN_QUELLE == "markdown" else STUDIEN_ROOT

# Lädt alle 16 Labels exakt so, wie sie in den anderen ML-Segmenten definiert sind
with open(CURRENT_FUSION_PATH / 'y_columns.json', 'r', encoding='utf-8') as f:
    CATEGORIES = json.load(f)

# Die Resultate werden direkt unter CURRENT_PREDICTIONS_PATH organisiert
TARGET_OUTPUT_ROOT = CURRENT_PREDICTIONS_PATH / PROMPT_TECHNIQUE / STRATEGIE_NAME

# Verzeichnisstruktur für die Jahre anlegen
for jahr in UNTERORDNER_JAHRE:
    (TARGET_OUTPUT_ROOT / jahr).mkdir(parents=True, exist_ok=True)

def clean_and_parse_json(raw_response):
    """Extrahiert robust das JSON-Array, selbst wenn das LLM Markdown-Tags nutzt."""
    text = raw_response.strip()
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*", "", text)
    
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return []
    return []

def classify_chunk_with_llm(text_content):
    """Sendet den Text an Ollama unter Verwendung eines Standard-Few-Shot-Prompts ohne Keyword-Hilfen."""
    
    # 🔄 GEÄNDERT: Keyword-Guidelines komplett entfernt. Der Prompt ist jetzt viel schlanker.
    prompt = f"""You are an expert reproducibility classifier trained to categorize research paper snippets across 16 dimensions.
    Analyze the given text chunk and determine which of these categories apply: {CATEGORIES}.
    IMPORTANT RULES:
    1. Multi-Label Classification is allowed for general papers.
    2. IF the text describes a "User Study" and you are confident it does not additionally describe a "software development", "hardware development", 
    "Literature Review", "Dataset Paper" or a "Algorithm development", focus primarily on "User Study" and "open_access". set all predictions of all 
    other labels as 0.according to our categories, the predicton for solely user studys should look like this: (0,0,0,1,0,0,0,x,0,0,0,0,0,0,0,0). x is the Entry
    for the open_access label.
    3. Return ONLY a valid JSON array of strings. No markdown, no filler.

    In the following, i will explain what the different labels mean.
    Algorithm Development: Main focus of the Study is presenting a novel algorithm for a known problem e.g. optimization techniques, machine learning models 
    or designing a pipeline (e.g. a machine learning/deep-learning pipeline).
    Hardware Development: finding a new or alternative solution for a known problem by constructing or designing novel hardware and testing it.
    Software Development: Publishing a novel framework, App or other software is the main contribution, often shared via a github URL.
    User Study: The paper primarily investigates human behavior, opinions, or interactions through experiments, surveys, interviews, or observational studies.
    Dataset Paper: The paper introduces a newly collected dataset that is explicitly released for use by other researchers.
    Literature Review: The paper systematically analyzes, surveys, or synthesizes findings from previously published research without introducing new experimental 
    results.
    is_dataset_paper: indicates if the study belongs to a special dataset-track of ACM. For this category to be true, the word "Dataset" 
    must always be a part of the Title. Example: "A Longitudinal Multimodal Dataset on [...]".
    open_access: Indicates whether the paper is freely accessible without paywalls.
    code_required: Indicates whether executing code is necessary to reproduce the main results. Often related to paper_type_categories software_development.
    code_available: Indicates the availability of the required code. Required software is often shared via a github URL.
    code_url: Indicates whether a URL to share the required software is specified.
    data_required: Indicates whether specific, not publicly available data is required to reproduce the results.
    data_available: Indicates whether the required data is made accessible via a URL e.g. over websites like OSF, kaggle, zenodo or github.
    data_url: Direct URL to the dataset (e.g. OSF,kaggle,zenodo, github).
    hardware_required: Indicates whether specialized or self assembled hardware is required to reproduce the results. Sensors or other cheap components should not
    be taken into account when predicting this label.
    software_required: Indicates whether proprietary, self built or non-open-source software is required to repdocuce the study's main findings.
    

    



    return exactly these keys, in the same order.
    ["Algorithm Development", "Hardware Development", "Software Development", "User Study", "Dataset Paper", "Literature Review", "is_dataset_paper",
    "open_access", "code_required", "code_available", "code_url", "data_required", "data_available", "data_url", "hardware_required", "software_required"]
    

    
    
    

    ---
    FEW-SHOT EXAMPLE 1 (Hardware, User Study & Artifacts):
    Input Text: "We designed a custom wearable sensor node using a flexible PCB layout. To evaluate the comfort, we recruited 15 participants for a user study. The dataset and firmware are available at https://github.com/project/core"
    Response: ["Hardware Development", "User Study", "code_required", "code_available", "code_url", "data_required", "data_available", "data_url", "hardware_required"]

    FEW-SHOT EXAMPLE 2 (Algorithm & Open Source):
    Input Text: "To minimize error, we introduce a novel loss function. The complete evaluation pipeline architecture has been uploaded to our public GitHub repository as an open-source library under open access terms."
    Response: ["Algorithm Development", "Software Development", "open_access", "code_required", "code_available", "code_url"]
    ---

    CRITICAL: You must respond ONLY with a valid JSON array of strings containing the matching categories from the allowed list. 
    Do not write any introduction, conversational filler, markdown formatting (like ```json), or explanations.

    Text chunk to analyze:
    {text_content}"""
    
    try:
        response = requests.post(OLLAMA_GENERATE_URL, json={
            "model": LLM_MODELL,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.0} 
        }, timeout=15)
        
        return clean_and_parse_json(response.json()['response'])
    except Exception:
        return []

# --- 2. MAIN PIPELINE ---
    print("🔍 Prüfe Verbindung zum Ollama-Server...")
if not check_ollama():
    print("❌ FEHLER: Keine Verbindung zum lokalen Ollama-Server!")
    print("💡 Bitte stelle sicher, dass die Ollama-App im Hintergrund läuft.")
    import sys
    sys.exit("Abbruch: Ollama-Server nicht erreichbar.")
else:
    print("✅ Ollama-Server ist erreichbar!")

print(f"🤖 Starte Llama 3.2 Multi-Label Klassifikation...")
print(f"📂 Modus: Erkennung basiert auf '{DATEN_QUELLE.upper()}'-Dateien (Suche nach *{DATEI_ENDUNG})")
print(f"🛠️ Technik: {PROMPT_TECHNIQUE}")
print(f"📍 Zielordner (CURRENT_PREDICTIONS_PATH): {TARGET_OUTPUT_ROOT}")

y_true_llm = []
y_pred_llm = []
processed_studies = 0
stop_processing = False

for jahr in UNTERORDNER_JAHRE:
    if stop_processing: break
    source_dir = QUELL_ORDNER_ROOT / jahr
    if not source_dir.exists(): continue
    
    studien_dateien = sorted([f for f in os.listdir(source_dir) if f.endswith(DATEI_ENDUNG)])
    
    for datei in studien_dateien:
        if processed_studies >= LLM_LIMIT:
            stop_processing = True
            break
            
        paper_id_key = datei.replace(DATEI_ENDUNG, "").strip()
        match = master_label_df[master_label_df['paper_id'] == paper_id_key]
        if match.empty: continue 
        
        out_json_path = TARGET_OUTPUT_ROOT / jahr / datei.replace(DATEI_ENDUNG, "_prediction.json")
        
        # Ground Truth für alle 16 Labels dynamisch extrahieren & flexibel parsen
        row = match.iloc[0]
        final_doc_true = []
        for cat in CATEGORIES:
            if cat in row:
                val = str(row[cat]).strip().lower()
                if val in ['1', '1.0', 'yes', 'true']:
                    final_doc_true.append(1)
                else:
                    final_doc_true.append(0)
            else:
                final_doc_true.append(1 if cat in str(row.to_dict()) else 0)
        
        # Fortsetzungs-Logik bei wiederholtem Start
        if out_json_path.exists():
            with open(out_json_path, 'r', encoding='utf-8') as f:
                saved_data = json.load(f)
                saved_vector = saved_data["predictions"]["binary_vector"]
            
            y_true_llm.append(final_doc_true)
            y_pred_llm.append(saved_vector)
            processed_studies += 1
            continue
        
        # NEUBERECHNUNG ÜBER DIE UNIVERSELLE EXTRAKTIONS-FUNKTION
        chunks, internal_meta = extract_pdf_structure(source_dir / datei)
        if not chunks: continue
        
        print(f"📄 Analysiere {datei} ({len(chunks)} Chunks)...")
        document_votes = {cat: 0 for cat in CATEGORIES}
        
        for chunk in tqdm(chunks, desc="Klassifiziere Chunks", leave=False):
            predicted_cats = classify_chunk_with_llm(chunk['text'])
            for cat in predicted_cats:
                if cat in document_votes:
                    document_votes[cat] += 1
                    
        # Prozent-Schwelle (Threshold)
        total_chunks = len(chunks)
        final_doc_pred = [0] * len(CATEGORIES)
        
        for i, cat in enumerate(CATEGORIES):
            votes_for_cat = document_votes[cat]
            if votes_for_cat > 0:
                activation_ratio = votes_for_cat / total_chunks
                if activation_ratio >= 0.25:  # 25% Schwelle für stabile Multi-Label-Erkennung
                    final_doc_pred[i] = 1
        
        # JSON im neuen, korrekten Pfad abspeichern
        with open(out_json_path, 'w', encoding='utf-8') as f:
            json.dump({
                "metadata": {
                    "source_filename": datei,
                    "extraction_meta": internal_meta,
                    "model": LLM_MODELL,
                    "prompt_technique": PROMPT_TECHNIQUE,
                    "strategy": STRATEGIE_NAME,
                    "timestamp": datetime.datetime.now().isoformat(),
                    "chunk_count": total_chunks,
                    "aggregation_strategy": "Percentage Threshold (>= 25% Chunk Frequency)"
                },
                "votes_distribution": document_votes,
                "predictions": {
                    "binary_vector": final_doc_pred,
                    "active_categories": [CATEGORIES[i] for i, v in enumerate(final_doc_pred) if v == 1]
                }
            }, f, indent=4, ensure_ascii=False)
            
        y_true_llm.append(final_doc_true)
        y_pred_llm.append(final_doc_pred)
        processed_studies += 1

# --- 3. EVALUIERUNG & METRIKEN ABSPEICHERN ---
if y_true_llm:
    # 1. Report als String generieren
    report_str = classification_report(np.array(y_true_llm), np.array(y_pred_llm), target_names=CATEGORIES, zero_division=0)
    
    # 2. In der Konsole ausgeben
    print(f"\n📊 DYNAMISCHER KLASSIFIZIERUNGS-BERICHT ({LLM_MODELL} - Quelle: {DATEN_QUELLE.upper()}):")
    print("-" * 65)
    print(report_str)
    print("-" * 65)
    
    # 3. Metriken als Datei direkt im TARGET_OUTPUT_ROOT ablegen (neben den Jahresordnern)
    metrics_filename = f"classification_metrics_{LLM_MODELL}_{DATEN_QUELLE}.txt"
    metrics_file_path = TARGET_OUTPUT_ROOT / metrics_filename
    
    with open(metrics_file_path, "w", encoding="utf-8") as f:
        f.write(f"=== KLASSIFIZIERUNGS-METRIKEN ===\n")
        f.write(f"Datum: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Modell: {LLM_MODELL}\n")
        f.write(f"Datenquelle: {DATEN_QUELLE}\n")
        f.write(f"Prompt Technik: {PROMPT_TECHNIQUE}\n")
        f.write(f"Anzahl evaluierter Studien: {len(y_true_llm)}\n")
        f.write("-" * 65 + "\n")
        f.write(report_str + "\n")
        f.write("-" * 65 + "\n")
        
    print(f"\n💾 Metriken erfolgreich gespeichert als Datei:")
    print(f"📂 Pfad: {metrics_file_path}")

✅ Ollama-Server ist erreichbar!
🤖 Starte Llama 3.2 Multi-Label Klassifikation...
📂 Modus: Erkennung basiert auf 'MARKDOWN'-Dateien (Suche nach *.md)
🛠️ Technik: few_shot_standard
📍 Zielordner (CURRENT_PREDICTIONS_PATH): /var/mnt/2TB/Dokumente/Meine_Bachelorarbeit/daten/predictions_llm/markdown/llama3.2/few_shot_standard/1200_chars_fixed_chunks
📄 Analysiere 2023_Dec_154.md (127 Chunks)...


Klassifiziere Chunks:   0%|          | 0/127 [00:00<?, ?it/s]


📊 DYNAMISCHER KLASSIFIZIERUNGS-BERICHT (llama3.2 - Quelle: MARKDOWN):
-----------------------------------------------------------------
                       precision    recall  f1-score   support

Algorithm Development       0.96      0.59      0.73       463
 Hardware Development       0.00      0.00      0.00       164
 Software Development       0.00      0.00      0.00       318
           User Study       0.59      1.00      0.74       353
        Dataset Paper       0.00      0.00      0.00        49
    Literature Review       0.00      0.00      0.00        11
     is_dataset_paper       0.00      0.00      0.00        15
          open_access       0.48      1.00      0.65       288
        code_required       1.00      0.01      0.01       521
       code_available       0.00      0.00      0.00       133
             code_url       0.00      0.00      0.00         0
        data_required       0.00      0.00      0.00       432
       data_available       0.00      0.00 